# 🏥 Pipeline de données — Prédiction du taux d'incidence
**Projet DataScientest / Liora — Direction de l'Actuariat Vie （CAAS）**

---

## Architecture : Schéma en étoile (Star Schema)

```
                              ┌────────────────────┐
                              │     dim_temps      │
                              │  PK: annee_mois    │
                              └─────────┬──────────┘
                                        │ annee_mois
                                        │
  ┌─────────────┐                       │                              ┌─────────────┐
  │ dim_geo_pop │                       ▼                              │   dim_csp   │
  │  PK: dept   ├──── dept ────►┌─────────────────────────┐◄─── dept ──┤  PK: dept   │
  └─────────────┘               │ fact_urgences           │            └─────────────┘
                                │                         │
                                │ FK: dept                │
                                │ FK: annee_mois          │
                                │                         │
                                │ taux_urgences_allergie  │
                                │ taux_urgences_asthme    │
                                │ taux_urgences_bronchio  │
                                │ taux_hosp_*             │
                                │ taux_sos_*              │
                                └───────────┬─────────────┘
                                            │ dept × annee_mois
                           ┌────────────────┼──────────────────┐
                           │                │                  │
                           ▼                ▼                  ▼
               ┌──────────────────┐ ┌──────────────┐ ┌──────────────────┐
               │    dim_meteo     │ │  dim_pollen  │ │ dim_qualite_air  │
               │ PK: dept×mois    │ │ PK: dept×mois│ │  PK: dept×mois   │
               └──────────────────┘ └──────────────┘ └──────────────────┘
```

> **Convention :** les flèches indiquent la direction de la jointure (FK → PK).  
> La table `fact_urgences` est la table centrale ; toutes les dimensions s'y joignent.

---

## Tables du modèle

| Table | Clé primaire (PK) | Granularité | Source | Lignes |
|---|---|---|---|---|
| `fact_urgences` | dept × annee_mois | Mois × Département | Santé Publique France | 6 912 |
| `dim_temps` | annee_mois | Mois | Calculé | 72 |
| `dim_meteo` | dept × annee_mois | Mois × Département | Météo France (MENSQ) | 6 912 |
| `dim_qualite_air` | dept × annee_mois | Mois × Département (2021–2025) | LCSQA / data.gouv.fr | ~5 516 |
| `dim_pollen` | dept × annee_mois | Mois × Département | RNSA | ~1 686 |
| `dim_geo_pop` | dept | Département | INSEE + Ameli | 96 |
| `dim_csp` | dept | Département | INSEE CSP | 96 |

---

## Flux de construction (dépendances)

```
  [Fichiers bruts]          [Fonctions build_*]         [Tables .parquet]
  ─────────────────         ──────────────────          ─────────────────

  (calculé)           ──►  build_dim_temps()       ──►  dim_temps.parquet
                                                           │
  allergie/asthme/    ──►  parse_urgences()         ──►  fact_urgences.parquet
  bronchiolite CSV                                         │
                                                           │
  MeteoFrance/MENSQ_* ──►  build_dim_meteo()        ──►  dim_meteo.parquet
       (+ split_corse_2a_2b pour la Corse)                                           │
                                                           │
  insee_demographie   ──►  build_dim_geo_pop()      ──►  dim_geo_pop.parquet
  + urbanisation                                           │
  + ameli_*.xls                                            │
                                                           │
  insee_csp.xlsx      ──►  build_dim_csp()          ──►  dim_csp.parquet
                                                           │
  AASQA/FR_E2_*.csv   ──►  build_station_dict_aasqa()──►  stations_aasqa_dept.parquet
       +                   build_dim_qualite_air()  ──►  dim_qualite_air.parquet
                                                           │
  BDD_daily/*.xls     ──►  build_dim_pollen()        ──►  dim_pollen.parquet
                                                           │
  (toutes les tables) ──►  build_model_view()        ──►  df_model.parquet ✅
```


---
## 0. Imports & configuration

In [1]:
# ── Import les bibliothèques nécessaires ─────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path
import gzip
import reverse_geocoder as rg
import re

In [ ]:
# ── Dossiers ─────────────────────────────────────────────────────────────────
# Le notebook vit dans notebooks/ mais les chemins sont relatifs à la racine
# du projet (convention cookiecutter-data-science) → on s'y replace si besoin.
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

RAW_DIR    = Path("data/raw")
TABLES_DIR = Path("data/processed")   # une table = un fichier .parquet
for d in [RAW_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Paramètres globaux ───────────────────────────────────────────────────────
ANNEE_DEBUT = 2020
ANNEE_FIN   = 2025

DEPTS = [f"{i:02d}" for i in range(1, 96) if i != 20] + ["2A", "2B"]
# Attention : pour Corse , il faut utiliser 2A et 2B, pas 20.
# Décommentez pour inclure les DOM :
# DEPTS += ["971", "972", "973", "974", "976"]

print(f"✅ Config : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"   Dossier tables : {TABLES_DIR.resolve()}")

---
## 1. ⏱️ `dim_temps` — Dimension temporelle

Entièrement calculée, aucun fichier à télécharger.

In [7]:
def build_dim_temps() -> pd.DataFrame:
    """
    Génère la dimension temporelle : une ligne par mois.
    Clé primaire : annee_mois (format 'YYYY-MM')
    """
    mois_range = pd.period_range(start=f"{ANNEE_DEBUT}-01",
                                  end=f"{ANNEE_FIN}-12", freq="M")
    df = pd.DataFrame({"annee_mois": mois_range.astype(str)})

    df["annee"]      = mois_range.year
    df["mois"]       = mois_range.month
    df["trimestre"]  = ((mois_range.month - 1) // 3) + 1
    df["semestre"]   = ((mois_range.month - 1) // 6) + 1

    # Encodage cyclique (évite la rupture déc → jan dans les modèles linéaires)
    df["sin_mois"]   = np.sin(2 * np.pi * df["mois"] / 12).round(6)
    df["cos_mois"]   = np.cos(2 * np.pi * df["mois"] / 12).round(6)

    # Indicateurs saisonniers (hémisphère nord)
    df["est_hiver"]     = df["mois"].isin([12, 1, 2]).astype(int)
    df["est_printemps"] = df["mois"].isin([3, 4, 5]).astype(int)
    df["est_ete"]       = df["mois"].isin([6, 7, 8]).astype(int)
    df["est_automne"]   = df["mois"].isin([9, 10, 11]).astype(int)

    # Saison pollinique (indicatif France métropolitaine)
    df["saison_pollen"] = df["mois"].isin([3, 4, 5, 6]).astype(int)

    # Flag Covid (peut influer sur les passages aux urgences)
    df["flag_covid"] = (
        ((df["annee"] == 2020) & (df["mois"] >= 3)) |
        (df["annee"] == 2021) |
        ((df["annee"] == 2022) & (df["mois"] <= 3))
    ).astype(int)

    print(f"dim_temps : {df.shape[0]} mois ({df['annee_mois'].iloc[0]} → {df['annee_mois'].iloc[-1]})")
    print(f"Colonnes  : {list(df.columns)}")
    return df


dim_temps = build_dim_temps()
dim_temps.to_parquet(TABLES_DIR / "dim_temps.parquet", index=False)
print(f"\n✅ Sauvegardé → {TABLES_DIR / 'dim_temps.parquet'}")
dim_temps.head(12)

dim_temps : 72 mois (2020-01 → 2025-12)
Colonnes  : ['annee_mois', 'annee', 'mois', 'trimestre', 'semestre', 'sin_mois', 'cos_mois', 'est_hiver', 'est_printemps', 'est_ete', 'est_automne', 'saison_pollen', 'flag_covid']

✅ Sauvegardé → data/tables/dim_temps.parquet


,annee_mois,annee,mois,trimestre,semestre,sin_mois,cos_mois,est_hiver,est_printemps,est_ete,est_automne,saison_pollen,flag_covid
0,2020-01,2020,1,1,1,0.500000,0.866025,1,0,0,0,0,0
1,2020-02,2020,2,1,1,0.866025,0.500000,1,0,0,0,0,0
2,2020-03,2020,3,1,1,1.000000,0.000000,0,1,0,0,1,1
3,2020-04,2020,4,2,1,0.866025,-0.500000,0,1,0,0,1,1
4,2020-05,2020,5,2,1,0.500000,-0.866025,0,1,0,0,1,1
5,2020-06,2020,6,2,1,0.000000,-1.000000,0,0,1,0,1,1
6,2020-07,2020,7,3,2,-0.500000,-0.866025,0,0,1,0,0,1
7,2020-08,2020,8,3,2,-0.866025,-0.500000,0,0,1,0,0,1
8,2020-09,2020,9,3,2,-1.000000,-0.000000,0,0,0,1,0,1
9,2020-10,2020,10,4,2,-0.866025,0.500000,0,0,0,1,0,1


---
## 2. 🎯 `fact_urgences` — Table de faits

**Source :** Santé Publique France — Odissé 

### Fichiers à télécharger → `data/raw/`
| Pathologie | URL | Nom fichier |
|---|---|---|
| Allergie | https://odisse.santepubliquefrance.fr/explore/dataset/allergie-passages-aux-urgences-et-actes-sos-medecins-dep/export/ | `allergie_urgences.csv` |
| Asthme | https://odisse.santepubliquefrance.fr/explore/dataset/asthme-passages-aux-urgences-et-actes-sos-medecins-dep/export/ | `asthme_urgences.csv` |
| Bronchiolite | https://odisse.santepubliquefrance.fr/explore/dataset/bronchiolite-passages-aux-urgences-et-actes-sos-medecins-departement/export/ | `bronchiolite_urgences.csv` |

> **Format export :** choisir **CSV avec séparateur `;`**

In [11]:
for fname in ["asthme_urgences.csv", "bronchiolite_urgences.csv", "allergie_urgences.csv"]:
    df = pd.read_csv(f"data/raw/{fname}", sep=",", nrows=1, dtype=str)
    print(f"\n{fname} :")
    for col in df.columns:
        print(f"  '{col}'")


asthme_urgences.csv :
  '1er jour de la semaine'
  'Semaine'
  'Département Code'
  'Département'
  'Classe d'âge'
  'Taux de passages aux urgences pour asthme'
  'Taux d'hospitalisations après passages aux urgences pour asthme'
  'Taux d'actes médicaux SOS médecins pour asthme'
  'Région Code'
  'Région'

bronchiolite_urgences.csv :
  '1er jour de la semaine'
  'Semaine'
  'Département Code'
  'Département'
  'Classe d'âge'
  'Taux de passages aux urgences pour bronchiolite'
  'Taux d'hospitalisations après passages aux urgences pour bronchiolite'
  'Taux d'actes médicaux SOS médecins pour bronchiolite'
  'Région Code'
  'Région'

allergie_urgences.csv :
  '1er jour de la semaine'
  'Semaine'
  'Département Code'
  'Département'
  'Classe d'âge'
  'Taux de passages aux urgences pour allergie'
  'Taux d'hospitalisations après passages aux urgences pour allergie'
  'Taux d'actes médicaux SOS médecins pour allergie'
  'Région Code'
  'Région'


In [8]:
df = pd.read_csv("data/raw/bronchiolite_urgences.csv", sep=",", dtype=str)
df = df.iloc[:, [0, 2, 4, 5, 6, 7]]
df.columns = ["date", "dept", "classe_age","taux_urgences", "taux_hosp", "taux_sos"]

# Voir toutes les valeurs uniques de classe_age
print(df["classe_age"].unique())

# il n'y a que 0 an dans la colonne classe_age, on peut filtrer dessus pour ne garder que les lignes "Tous âges"

['0 an']


In [ ]:
def parse_urgences(pathologie: str, filepath: Path) -> pd.DataFrame:
    """
    Charge et nettoie un fichier CSV de Santé Publique France.

    PARAMÈTRES :
    ─────────────
    pathologie : "allergie", "asthme" ou "bronchiolite"
    filepath   : chemin vers le fichier CSV brut

    RETOUR :
    ─────────
    DataFrame avec colonnes :
        dept                        → code département (ex: "75")
        annee_mois                  → période (ex: "2021-03")
        taux_urgences_{pathologie}  → nb de passages aux urgences pour la pathologie,
                                      pour 100 000 passages aux urgences avec un diagnostic
                                      codé (toutes causes confondues), même dept/classe d'âge/semaine
        taux_hosp_{pathologie}      → nb d'hospitalisations après passage aux urgences pour la
                                      pathologie, pour 100 000 hospitalisations après passage avec
                                      diagnostic renseigné (toutes causes). Mesure la part de la
                                      pathologie parmi les hospitalisations post-urgences, PAS le
                                      risque d'hospitalisation parmi les patients de la pathologie
        taux_sos_{pathologie}       → nb d'actes SOS Médecins pour la pathologie, pour 100 000
                                      actes SOS Médecins avec diagnostic renseigné (circuit de
                                      soins différent : médecine de ville/domicile, non hospitalier)

    ⚠️  IMPORTANT — Ces 3 taux sont des PARTS RELATIVES (dénominateur = volume total
    toutes causes confondues sur le même sous-groupe), PAS des taux d'incidence en
    population. Santé Publique France précise explicitement qu'ils ne doivent être
    NI agrégés entre départements NI agrégés entre classes d'âge, car le dénominateur
    diffère d'un sous-groupe à l'autre (ex : moyenne de taux_urgences entre deux depts
    n'a pas de sens). Le pipeline respecte déjà cette contrainte : dept reste une clé
    de la table (jamais moyenné) et un seul niveau de classe d'âge est conservé par
    pathologie (cf ÉTAPE 3). Ne pas casser cette invariance dans du code ultérieur.

    STRATÉGIE :
    ─────────────
    On utilise les POSITIONS des colonnes (iloc) plutôt que leurs noms,
    car les fichiers mélangent guillemets simples et doubles ce qui
    rend toute comparaison de chaînes instable.

    Structure commune aux 3 fichiers :
        [0] 1er jour de la semaine  ← date
        [1] Semaine
        [2] Département Code        ← dept
        [3] Département
        [4] Classe d'âge            ← filtre age
        [5] Taux urgences           ← Y principal
        [6] Taux hospitalisations   ← Y secondaire
        [7] Taux SOS médecins       ← Y tertiaire
        [8] Région Code
        [9] Région
    """

    # ── ÉTAPE 1 : Chargement ─────────────────────────────────────────────────
    # dtype=str → tout en texte d'abord pour éviter les erreurs de type mixte
    df = pd.read_csv(filepath, sep=",", encoding="utf-8",
                     dtype=str, low_memory=False)
    print(f"\n[{pathologie.upper()}]")
    print(f"  Lignes brutes       : {len(df):,}")

    # ── ÉTAPE 2 : Sélection et renommage par position ────────────────────────
    # On prend uniquement les 8 premières colonnes utiles
    # et on leur donne des noms clairs tout de suite
    df = df.iloc[:, [0, 2, 4, 5, 6, 7]].copy()
    df.columns = [
        "date",
        "dept",
        "classe_age",
        f"taux_urgences_{pathologie}",
        f"taux_hosp_{pathologie}",
        f"taux_sos_{pathologie}",
    ]

    # ── ÉTAPE 3 : Filtrage sur la classe d'âge globale ───────────────────────
    # Allergie & Asthme  → "Tous âges"  (population complète)
    # Bronchiolite       → "0 an"       (nourrissons uniquement, pas de "Tous âges")
    # On garde les deux valeurs possibles avec isin()
    # ⚠️  Un seul niveau de classe d'âge conservé par pathologie : ne jamais
    # mélanger/agréger "Tous âges" avec "0 an" ou avec les autres tranches
    # (00-14, 15-64, 65+), les dénominateurs ne sont pas comparables entre elles.
    valeurs_age = df["classe_age"].unique().tolist()
    print(f"  Classes d'âge dispo : {valeurs_age}")

    valeurs_valides = ["Tous âges", "0 an"]
    df = df[df["classe_age"].isin(valeurs_valides)].copy()
    print(f"  Après filtre âge    : {len(df):,} lignes "
          f"({df['classe_age'].unique().tolist()})")

    # ── ÉTAPE 4 : Parsing de la date ─────────────────────────────────────────
    # La colonne contient le lundi de chaque semaine : "2020-03-30"
    # On la convertit pour extraire l'année et le mois
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    nb_invalides = df["date"].isna().sum()
    if nb_invalides > 0:
        print(f"  ⚠️  {nb_invalides} dates invalides supprimées")
    df = df.dropna(subset=["date"])

    # annee_mois = clé de jointure avec toutes les autres tables
    # Ex : semaines S13, S14, S15 de mars → toutes donnent "2020-03"
    df["annee_mois"] = df["date"].dt.to_period("M").astype(str)
    df["annee"]      = df["date"].dt.year

    # ── ÉTAPE 5 : Filtrage temporel 2020–2025 ────────────────────────────────
    avant = len(df)
    df = df[(df["annee"] >= ANNEE_DEBUT) & (df["annee"] <= ANNEE_FIN)]
    print(f"  Filtrage {ANNEE_DEBUT}–{ANNEE_FIN}      : {avant:,} → {len(df):,} lignes")

    # ── ÉTAPE 6 : Nettoyage du code département ───────────────────────────────
    # "9" → "09" | " 68" → "68" | "2a" → "2A"
    df["dept"] = (
        df["dept"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.zfill(2)
    )
    avant = len(df)
    df = df[df["dept"].isin(DEPTS)]
    print(f"  Filtrage depts      : {avant:,} → {len(df):,} lignes")

    # ── ÉTAPE 7 : Conversion numérique des taux ───────────────────────────────
    # Les valeurs étaient en texte (dtype=str au chargement)
    # errors="coerce" → les "-" ou "n/a" deviennent NaN
    cols_taux = [f"taux_urgences_{pathologie}",
                 f"taux_hosp_{pathologie}",
                 f"taux_sos_{pathologie}"]
    for col in cols_taux:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # ── ÉTAPE 8 : Agrégation hebdomadaire → mensuelle ─────────────────────────
    # Les données sont hebdomadaires (1 ligne par semaine × département)
    # On prend la MOYENNE des semaines du mois car ce sont des TAUX
    # (pas des comptages → la somme n'aurait pas de sens)
    # ⚠️  Cette moyenne reste PAR département (dept fait partie du groupby) :
    # on ne moyenne jamais entre départements, seulement entre semaines d'un
    # même mois pour un même département.
    df_agg = (
        df.groupby(["dept", "annee_mois"])[cols_taux]
        .mean()
        .round(2)
        .reset_index()
    )

    print(f"  ✅ Résultat         : {df_agg.shape[0]:,} lignes × {df_agg.shape[1]} colonnes")
    print(f"  Période             : {df_agg['annee_mois'].min()} → {df_agg['annee_mois'].max()}")
    print(f"  Départements        : {df_agg['dept'].nunique()}")
    return df_agg


# ══════════════════════════════════════════════════════════════════════════════
# CHARGEMENT DES 3 PATHOLOGIES
# ══════════════════════════════════════════════════════════════════════════════

FICHIERS_URGENCES = {
    "allergie":     "allergie_urgences.csv",
    "asthme":       "asthme_urgences.csv",
    "bronchiolite": "bronchiolite_urgences.csv",
}

dfs_patho = {}
for patho, fname in FICHIERS_URGENCES.items():
    fpath = RAW_DIR / fname
    if fpath.exists():
        dfs_patho[patho] = parse_urgences(patho, fpath)
    else:
        print(f"⚠️  Fichier manquant : {fname}")


# ══════════════════════════════════════════════════════════════════════════════
# FUSION EN UNE SEULE TABLE fact_urgences
# ══════════════════════════════════════════════════════════════════════════════

if dfs_patho:

    # Squelette complet : 96 depts × 72 mois = 6 912 lignes garanties
    # Même si un département n'a pas de données → ligne présente avec NaN
    dim_t = pd.read_parquet(TABLES_DIR / "dim_temps.parquet")[["annee_mois"]]
    skeleton = pd.MultiIndex.from_product(
        [DEPTS, dim_t["annee_mois"]],
        names=["dept", "annee_mois"]
    ).to_frame(index=False)

    # Jointure de chaque pathologie sur le squelette
    fact = skeleton.copy()
    for patho, df_tmp in dfs_patho.items():
        fact = fact.merge(df_tmp, on=["dept", "annee_mois"], how="left")

    # Sauvegarde
    fact.to_parquet(TABLES_DIR / "fact_urgences.parquet", index=False)
    print(f"\n✅ fact_urgences sauvegardé")
    print(f"   {fact.shape[0]:,} lignes × {fact.shape[1]} colonnes")
    display(fact.head(10))

    # Rapport de couverture
    print("\n📊 Couverture finale :")
    for col in [c for c in fact.columns if c.startswith("taux_")]:
        pct = fact[col].notna().mean() * 100
        statut = "✅" if pct > 80 else "⚠️ "
        barre = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        print(f"  {statut} {col:<40} {barre} {pct:.1f}%")

---
## 3. 🌡️ `dim_meteo` — Dimension météorologique

**Source :** Météo France — Données climatologiques de base **mensuelles** (data.gouv.fr)
URL : https://www.data.gouv.fr/datasets/donnees-climatologiques-de-base-mensuelles

> ⚠️ **Changement de source.** L'ancienne source (archives SYNOP horaires) ne comptait
> que 42 stations en France métropolitaine (41 départements couverts) → 55 départements sur 96 sans aucune donnée
> météo (58 % des lignes de `df_model` avec `temp_moy` = NaN). On la remplace par le
> réseau climatologique de base de Météo France, un fichier par département, beaucoup
> plus dense (déjà 20+ stations rien que pour le département 01) et **déjà agrégé au
> mois par station** — plus besoin de ré-agréger des relevés horaires.

### Fichiers à télécharger → `data/raw/MeteoFrance/`
Pour chaque département (codes `01` à `95` ; la Corse est fournie sous un seul code
`20`, non séparée en 2A/2B — voir plus bas) :
- `MENSQ_XX_previous-1950-2024.csv.gz`
- `MENSQ_XX_latest-2025-2026.csv.gz`

URL de base : `https://meteofrance.s3.sbg.io.cloud.ovh.net/data/synchro_ftp/BASE/MENS/`
Description des champs : `MENSQ_descriptif_champs.csv` (même dossier)

### Correspondance des variables
| Colonne `dim_meteo` | Champ MENSQ | Signification |
|---|---|---|
| `temp_moy` | `TM` | Température moyenne mensuelle (°C) |
| `temp_min` | `TN` | Moyenne mensuelle des températures minimales (°C) |
| `temp_max` | `TX` | Moyenne mensuelle des températures maximales (°C) |
| `humidite_moy` | `UMM` | Humidité relative moyenne (%) |
| `vent_moy` | `FFM` | Vitesse moyenne du vent à 10 m (m/s) |
| `precip_total` | `RR` | Cumul mensuel des précipitations (mm) |

### ⚠️ Cas particulier : la Corse
Ce jeu de données ne distingue pas 2A/2B : toutes les stations corses sont sous le
code `20`. On les sépare en 2A (Corse-du-Sud) / 2B (Haute-Corse) via un reverse
geocoding sur leur latitude/longitude (même logique que celle utilisée pour les
stations SYNOP auparavant), uniquement pour ce département.

In [29]:
# Vérifier la structure d'un fichier MENSQ (1 fichier = 1 département)
df_check = pd.read_csv(
    "data/raw/MeteoFrance/MENSQ_01_previous-1950-2024.csv.gz",
    sep=";", compression="gzip", nrows=5)
print("=== MENSQ (dept 01) ===")
print(df_check.columns.tolist())
display(df_check[["NUM_POSTE", "NOM_USUEL", "LAT", "LON", "AAAAMM",
                   "TM", "TX", "TN", "UMM", "FFM", "RR"]].head())

=== MENSQ (dept 01) ===
['NUM_POSTE', 'NOM_USUEL', 'LAT', 'LON', 'ALTI', 'AAAAMM', 'RR', 'QRR', 'NBRR', 'RR_ME', 'RRAB', 'QRRAB', 'RRABDAT', 'NBJRR1', 'NBJRR5', 'NBJRR10', 'NBJRR30', 'NBJRR50', 'NBJRR100', 'PMERM', 'QPMERM', 'NBPMERM', 'PMERMINAB', 'QPMERMINAB', 'PMERMINABDAT', 'TX', 'QTX', 'NBTX', 'TX_ME', 'TXAB', 'QTXAB', 'TXDAT', 'TXMIN', 'QTXMIN', 'TXMINDAT', 'NBJTX0', 'NBJTX25', 'NBJTX30', 'NBJTX35', 'NBJTXI20', 'NBJTXI27', 'NBJTXS32', 'TN', 'QTN', 'NBTN', 'TN_ME', 'TNAB', 'QTNAB', 'TNDAT', 'TNMAX', 'QTNMAX', 'TNMAXDAT', 'NBJTN5', 'NBJTN10', 'NBJTNI10', 'NBJTNI15', 'NBJTNI20', 'NBJTNS20', 'NBJTNS25', 'NBJGELEE', 'TAMPLIM', 'QTAMPLIM', 'TAMPLIAB', 'QTAMPLIAB', 'TAMPLIABDAT', 'NBTAMPLI', 'TM', 'QTM', 'NBTM', 'TMM', 'QTMM', 'NBTMM', 'NBJTMS24', 'TMMIN', 'QTMMIN', 'TMMINDAT', 'TMMAX', 'QTMMAX', 'TMMAXDAT', 'UNAB', 'QUNAB', 'UNABDAT', 'NBUN', 'UXAB', 'QUXAB', 'UXABDAT', 'NBUX', 'UMM', 'QUMM', 'NBUM', 'TSVM', 'QTSVM', 'NBTSVM', 'ETP', 'QETP', 'FXIAB', 'QFXIAB', 'DXIAB', 'QDXIAB', 'FXIDA

,NUM_POSTE,NOM_USUEL,LAT,LON,AAAAMM,TM,TX,TN,UMM,FFM,RR
0,1010001,ANGLEFORT,45.913667,5.809833,195001,NaN,NaN,NaN,NaN,NaN,49.5
1,1010001,ANGLEFORT,45.913667,5.809833,195002,NaN,NaN,NaN,NaN,NaN,237.6
2,1010001,ANGLEFORT,45.913667,5.809833,195003,NaN,NaN,NaN,NaN,NaN,23.8
3,1010001,ANGLEFORT,45.913667,5.809833,195005,NaN,NaN,NaN,NaN,NaN,83.6
4,1010001,ANGLEFORT,45.913667,5.809833,195006,NaN,NaN,NaN,NaN,NaN,75.3


In [30]:
def split_corse_2a_2b(df_20: pd.DataFrame) -> dict:
    """
    Le fichier MENSQ_20_* regroupe toutes les stations de Corse sous un seul
    code département "20", sans distinction 2A (Corse-du-Sud) / 2B (Haute-Corse).

    On identifie le bon département pour chaque station via un reverse
    geocoding sur sa latitude/longitude (même approche que pour les stations
    SYNOP dans la version précédente du pipeline).

    Retourne : dict { NUM_POSTE (int) → "2A" ou "2B" }
    """
    stations = df_20[["NUM_POSTE", "LAT", "LON"]].drop_duplicates().copy()

    coords = list(zip(stations["LAT"], stations["LON"]))
    resultats = rg.search(coords)
    dept_nom = [r["admin2"] for r in resultats]

    DEPT_NOM_TO_CODE = {
        "Departement de la Corse-du-Sud": "2A",
        "Departement de la Haute-Corse": "2B",
    }
    stations["dept"] = [DEPT_NOM_TO_CODE.get(n) for n in dept_nom]

    non_resolues = stations["dept"].isna().sum()
    if non_resolues:
        print(f"  ⚠️  {non_resolues} stations corses non résolues (ignorées)")
    else:
        print(f"  ✅ {len(stations)} stations corses réparties en 2A/2B")

    return dict(zip(stations["NUM_POSTE"], stations["dept"]))

Notre variable cible — les passages aux urgences — est elle-même mensuelle. Le modèle a besoin de capturer des dynamiques saisonnières fines, par exemple :

Mars 2021 : début de la saison pollinique + remontée des températures → pic de passages aux urgences pour allergie
Décembre 2021 : chute des températures → augmentation des bronchiolites

Contrairement aux anciennes archives SYNOP (relevés toutes les 3h, qu'il fallait
ré-agréger au mois), les fichiers MENSQ de Météo France sont **déjà agrégés au mois,
par station**. Il ne reste donc qu'à agréger les différentes stations d'un même
département sur un même mois (moyenne) — pas de ré-agrégation temporelle à faire.

In [ ]:
def build_dim_meteo() -> pd.DataFrame:
    """
    Construit la table dim_meteo à partir des fichiers mensuels MENSQ de
    Météo France (un fichier par département, déjà agrégé au mois par station).

    STRATÉGIE :
    ───────────
    1. Charger, pour chaque département 01-95 (dont "20" pour la Corse),
       les fichiers "previous-1950-2024" et "latest-2025-2026"
    2. Filtrer sur la période ANNEE_DEBUT-ANNEE_FIN
    3. Cas particulier "20" → séparer les stations en 2A/2B (reverse geocoding,
       cf. split_corse_2a_2b ci-dessus)
    4. Renommer les champs MENSQ vers les noms de colonnes du schéma existant
    5. Agréger les stations d'un même département/mois → moyenne
    6. Sauvegarder en parquet

    VARIABLES PRODUITES : cf. tableau de correspondance dans la cellule markdown
    ci-dessus (temp_moy, temp_min, temp_max, humidite_moy, vent_moy, precip_total)

    CLÉ PRIMAIRE : dept × annee_mois
    """
    meteo_dir = RAW_DIR / "MeteoFrance"

    cols_utiles = ["NUM_POSTE", "LAT", "LON", "AAAAMM",
                   "TM", "TX", "TN", "UMM", "FFM", "RR"]
    RENAME_MAP = {
        "TM": "temp_moy", "TX": "temp_max", "TN": "temp_min",
        "UMM": "humidite_moy", "FFM": "vent_moy", "RR": "precip_total",
    }

    # ── ÉTAPE 1 : Charger chaque département (01-95, incl. "20" pour Corse) ──
    dfs = []
    df_20 = None  # traité à part le temps de la scission 2A/2B

    for code in [f"{i:02d}" for i in range(1, 96)]:
        frames = []
        for periode in ["previous-1950-2024", "latest-2025-2026"]:
            fpath = meteo_dir / f"MENSQ_{code}_{periode}.csv.gz"
            if not fpath.exists():
                continue
            frames.append(pd.read_csv(
                fpath, sep=";", compression="gzip",
                usecols=lambda c: c in cols_utiles, low_memory=False
            ))
        if not frames:
            print(f"  ⚠️  Aucun fichier pour le département {code}")
            continue

        df_code = pd.concat(frames, ignore_index=True)
        df_code["annee"] = df_code["AAAAMM"] // 100
        df_code = df_code[(df_code["annee"] >= ANNEE_DEBUT) & (df_code["annee"] <= ANNEE_FIN)]

        if code == "20":
            df_20 = df_code
        else:
            df_code = df_code.copy()
            df_code["dept"] = code
            dfs.append(df_code)

    # ── ÉTAPE 2 : Cas particulier de la Corse (scission 2A/2B) ───────────────
    if df_20 is not None and not df_20.empty:
        station_to_dept_corse = split_corse_2a_2b(df_20)
        df_20 = df_20.copy()
        df_20["dept"] = df_20["NUM_POSTE"].map(station_to_dept_corse)
        df_20 = df_20.dropna(subset=["dept"])
        dfs.append(df_20)

    if not dfs:
        print("❌ Aucun fichier MENSQ trouvé dans data/raw/MeteoFrance/")
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)
    print(f"  Total brut {ANNEE_DEBUT}-{ANNEE_FIN} (tous départements) : {len(df):,} lignes")

    # ── ÉTAPE 3 : annee_mois = clé de jointure avec les autres tables ────────
    aaaamm_str = df["AAAAMM"].astype(str)
    df["annee_mois"] = aaaamm_str.str[:4] + "-" + aaaamm_str.str[4:6]

    # ── ÉTAPE 4 : Conversion numérique + renommage vers le schéma existant ──
    for col in RENAME_MAP:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.rename(columns=RENAME_MAP)

    # ── ÉTAPE 5 : Agrégation des stations d'un même dept/mois → moyenne ─────
    # (chaque valeur MENSQ est déjà une moyenne mensuelle PAR STATION ;
    # ici on moyenne simplement entre les stations d'un même département)
    cols_sortie = list(RENAME_MAP.values())
    df_agg = (
        df.groupby(["dept", "annee_mois"])[cols_sortie]
        .mean()
        .round(2)
        .reset_index()
    )

    print(f"  ✅ dim_meteo : {df_agg.shape[0]:,} lignes × {df_agg.shape[1]} colonnes")
    print(f"  Période      : {df_agg['annee_mois'].min()} → {df_agg['annee_mois'].max()}")
    print(f"  Départements : {df_agg['dept'].nunique()} / {len(DEPTS)} couverts")
    return df_agg


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════

print("Construction de dim_meteo...")
dim_meteo = build_dim_meteo()

if not dim_meteo.empty:
    dim_meteo.to_parquet(TABLES_DIR / "dim_meteo.parquet", index=False)
    print(f"\n✅ Sauvegardé → data/processed/dim_meteo.parquet")
    display(dim_meteo.head(10))

In [32]:
# Vérification de couverture : plus aucun département manquant ?
depts_manquants = sorted(set(DEPTS) - set(dim_meteo["dept"].unique()))
print(f"Départements sans aucune donnée météo : {depts_manquants or 'aucun ✅'}")

doublons = dim_meteo.duplicated(subset=["dept", "annee_mois"]).sum()
print(f"Doublons (dept, annee_mois) : {doublons}")

print("\nTaux de valeurs manquantes par colonne :")
print((dim_meteo.isna().mean() * 100).round(1))

Départements sans aucune donnée météo : aucun ✅
Doublons (dept, annee_mois) : 0

Taux de valeurs manquantes par colonne :
dept            0.0
annee_mois      0.0
temp_moy        0.0
temp_max        0.0
temp_min        0.0
humidite_moy    3.1
vent_moy        3.1
precip_total    0.0
dtype: float64


---
## 4. 🗺️ `dim_geo_pop` — Dimension géographie & population

Fusionne 3 sources indépendantes, chacune apportant des variables différentes.
Table finale : **1 ligne par département** (pas de dimension temporelle — ce sont
des variables structurelles, considérées stables sur la période 2020-2025).

### Fichiers à télécharger → `data/raw/geo_pop/`
| Fichier | Source | URL | Période couverte |
|---|---|---|---|
| `insee_demographie.csv` | CNAM — Cartographie des pathologies (effectifs par pathologie/âge/dept) | https://www.data.gouv.fr/datasets/pathologies-effectif-de-patients-par-pathologie-sexe-classe-dage-et-territoire-departement-region | 2015-2024, mise à jour annuelle |
| `insee_urbanisation.xlsx` | INSEE — Population par taille d'unité urbaine | https://www.insee.fr/fr/statistiques/5039853 | 2017 (zonage "unités urbaines 2020", mais basé sur le recensement 2017 — voir note ci-dessous) |
| `ameli_acces_soins_YYYY.xls` (2020 à 2024) | Ameli — Densité des professionnels de santé libéraux | https://www.assurance-maladie.ameli.fr/etudes-et-donnees/densite-professionnels-sante-liberaux-departement | 1 fichier par année, 2020-2024 (2024 = dernière année disponible, pas de 2025 à ce jour) |

### Variables produites
| Colonne | Source | Champ d'origine | Notes |
|---|---|---|---|
| `pop_totale` | INSEE démographie | `Npop`, filtré sur `cla_age_5 == "tsage"` | Dernière année dispo (dynamique, actuellement 2024). ⚠️ Ne pas sommer `Npop` sur tous les `cla_age_5` : la ligne "tsage" est déjà l'agrégat "tous âges" — la sommer avec les tranches d'âge individuelles compte la population deux fois (bug corrigé). |
| `part_seniors` | INSEE démographie | `Npop` des tranches 65 ans et + / `pop_totale` | |
| `part_jeunes` | INSEE démographie | `Npop` des tranches < 15 ans / `pop_totale` | |
| `prev_resp_chronique` | INSEE démographie | `prev` (déjà calculé), filtré sur `patho_niv1 == "Maladies respiratoires chroniques (hors mucoviscidose)"` et `cla_age_5 == "tsage"` | Prévalence en population, proxy de vulnérabilité respiratoire de fond. ⚠️ Catégorie CNAM large (pas uniquement l'asthme, pas de sous-catégorie dédiée) ; **aucune catégorie équivalente n'existe pour allergie ou bronchiolite** dans cette nomenclature — ce n'est pas un substitut aux `taux_urgences_*`, seulement une variable explicative complémentaire. |
| `tx_urbain` | INSEE urbanisation | Part de la population en unité urbaine (%) | Donnée figée à 2017 : c'est la plus récente disponible — le zonage "unités urbaines 2020" (le plus à jour publié par l'INSEE) est lui-même recalculé sur la population du recensement 2017, il n'existe pas de version plus récente à l'échelle département. |
| `densite_med_gen` | Ameli | `DENSITE /100 000 hab.`, feuille "Généralistes et MEP" | Moyenne 2020-2024 (5 fichiers annuels) → une valeur fixe par département, pas de variation temporelle dans le modèle. |
| `densite_spe` | Ameli | `DENSITE /100 000 hab.`, feuille "Spécialistes" | Idem, moyenne 2020-2024. |

> **Pourquoi la densité médicale (Ameli) ?** Un département où l'accès aux généralistes
> est faible pousse mécaniquement plus de patients vers les urgences pour des motifs
> qui auraient pu être traités en médecine de ville — un biais structurel du système
> de soins, indépendant de la gravité réelle de la pathologie. Inclure cette variable
> aide le modèle à distinguer "plus d'urgences car plus de maladie" de "plus d'urgences
> car moins d'accès aux soins de ville".


In [33]:
# Check les noms des variables dans les bases de données INSEE et Ameli

# INSEE démographie
df = pd.read_csv("data/raw/geo_pop/insee_demographie.csv", sep=";", nrows=3)
print("=== INSEE démographie ===")
print(df.columns.tolist())
display(df.head(2))

# INSEE urbanisation
df2 = pd.read_excel(
    "data/raw/geo_pop/insee_urbanisation.xlsx",
    sheet_name=1,
    header=None,
    skiprows=2  # sauter les 2 premières lignes de métadonnées
)
print("\n=== INSEE urbanisation ===")
display(df2.head(10))

# Ameli (une année) : plusieurs sheets 
df3 = pd.read_excel("data/raw/geo_pop/ameli_acces_soins_2022.xls", nrows=3)
print("\n=== Ameli 2022 ===")
print(df3.columns.tolist())
display(df3.head(2)) 

=== INSEE démographie ===
['annee', 'patho_niv1', 'patho_niv2', 'patho_niv3', 'top', 'cla_age_5', 'sexe', 'region', 'dept', 'Ntop', 'Npop', 'prev', 'Niveau prioritaire', 'libelle_classe_age', 'libelle_sexe', 'tri']


,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri
0,2015,Maladies du foie ou du pancréas (hors mucovisc...,Maladies du foie ou du pancréas (hors mucovisc...,Maladies du foie ou du pancréas (hors mucovisc...,MFP_CAT_EXC,00-04,9,44,57,30,49550,0.054,"1,2,3",de 0 à 4 ans,tous sexes,6.0
1,2015,Maladies du foie ou du pancréas (hors mucovisc...,Maladies du foie ou du pancréas (hors mucovisc...,Maladies du foie ou du pancréas (hors mucovisc...,MFP_CAT_EXC,00-04,9,52,44,30,81200,0.038,"1,2,3",de 0 à 4 ans,tous sexes,6.0



=== INSEE urbanisation ===


,0,1,2
0,01,Ain,67.0
1,02,Aisne,53.2
2,03,Allier,58.3
3,04,Alpes-de-Haute-Provence,61.9
4,05,Hautes-Alpes,59.5
5,06,Alpes-Maritimes,95.9
6,07,Ardèche,63.3
7,08,Ardennes,57.3
8,09,Ariège,55.5
9,10,Aube,61.3



=== Ameli 2022 ===
['Année : ', 2022]


,Année :,2022
0,NaN,NaN
1,Sources :,les données de ce fichier sont issues de Amos ...


In [ ]:
def build_dim_geo_pop() -> pd.DataFrame:
    """
    Construit la table dim_geo_pop en fusionnant 3 sources :
      1. INSEE démographie  → population totale, structure par âge
      2. INSEE urbanisation → taux d'urbanisation par département
      3. Ameli (2020-2024)  → densité de professionnels de santé

    CLÉ PRIMAIRE : dept (une ligne par département)

    COLONNES PRODUITES :
    ────────────────────
    dept              → code département (ex: "75")
    pop_totale        → population totale
    part_seniors      → part des 65 ans et plus
    part_jeunes       → part des moins de 15 ans
    tx_urbain         → taux de population en zone urbaine (%)
    densite_med_gen   → densité médecins généralistes /100k hab
    densite_spe       → densité médecins spécialistes /100k hab
    prev_resp_chronique → prévalence des maladies respiratoires chroniques
                          (hors mucoviscidose) en population, dernière année
                          disponible (proxy de vulnérabilité respiratoire de
                          fond ; ne pas confondre avec taux_urgences_asthme,
                          qui mesure une part d'activité aux urgences)
    """

    geo_pop_dir = RAW_DIR / "geo_pop"

    # ══════════════════════════════════════════════════════════════════
    # SOURCE 1 : INSEE démographie
    # ══════════════════════════════════════════════════════════════════
    # Ce fichier contient les effectifs de patients par pathologie,
    # âge et département. On l'utilise pour extraire la structure
    # démographique (population par tranche d'âge).
    #
    # Colonnes utiles :
    #   annee      → année
    #   dept       → code département
    #   cla_age_5  → classe d'âge (00-04, 05-09, ..., 85+)
    #   Npop       → population de référence pour ce groupe
    #   sexe       → 9 = tous sexes confondus

    print("Chargement INSEE démographie...")
    df_demo = pd.read_csv(
        geo_pop_dir / "insee_demographie.csv",
        sep=";", encoding="utf-8", low_memory=False
    )
    print(f"  Brut : {len(df_demo):,} lignes")

    # Garder uniquement : tous sexes (sexe=9) + dernière année dispo
    df_demo = df_demo[df_demo["sexe"] == 9].copy()
    derniere_annee = df_demo["annee"].max()
    df_demo = df_demo[df_demo["annee"] == derniere_annee]
    print(
        f"  Après filtre (sexe=9, annee={derniere_annee}) : {len(df_demo):,} lignes")

    # Nettoyage département
    df_demo["dept"] = (
        df_demo["dept"].astype(str).str.strip().str.upper().str.zfill(2)
    )

    # Conversion numérique
    df_demo["Npop"] = pd.to_numeric(df_demo["Npop"], errors="coerce")
    df_demo["cla_age_5"] = df_demo["cla_age_5"].astype(str).str.strip()

    # ── Prévalence des maladies respiratoires chroniques ─────────────────
    # Même fichier, mais un patho_niv1 différent ("Total consommants..."
    # ci-dessous sert uniquement à la population totale). On l'extrait
    # AVANT le filtre patho_niv1 suivant, qui restreint df_demo.
    # Proxy de vulnérabilité respiratoire de fond de la population —
    # à ne pas confondre avec taux_urgences_asthme (part d'activité aux
    # urgences, pas une prévalence en population, cf. section fact_urgences).
    df_resp = df_demo[
        (df_demo["patho_niv1"] == "Maladies respiratoires chroniques (hors mucoviscidose)")
        & (df_demo["cla_age_5"] == "tsage")
    ][["dept", "prev"]].rename(columns={"prev": "prev_resp_chronique"})
    df_resp["prev_resp_chronique"] = pd.to_numeric(df_resp["prev_resp_chronique"], errors="coerce")
    print(f"  ✅ Prévalence respiratoire chronique : {len(df_resp)} départements (année {derniere_annee})")

    # Problème : Le fichier contient une ligne par combinaison pathologie × tranche d'âge × département, ce qui entraîne un comptage multiple de Npop lors de l'agrégation.
    # Solution : On filtre sur patho_niv1 == "Total consommants tous régimes", qui représente la population totale par tranche d'âge sans doublon pathologique.

    # Garder uniquement "Total consommants tous régimes"
    # → évite le double-comptage par pathologie
    df_demo = df_demo[
        df_demo["patho_niv1"] == "Total consommants tous régimes"
    ].copy()
    print(f"  Après filtre patho totale : {len(df_demo):,} lignes")

    # Population totale par département
    # ⚠️  cla_age_5 == "tsage" est DÉJÀ l'agrégat "tous âges" fourni par le
    # fichier. Sommer Npop sur tous les groupes d'âge (00-04, 05-09, ...)
    # EN PLUS de la ligne "tsage" compte la population totale deux fois
    # (bug précédent : pop_totale ≈ 2x la population réelle, ex. dept 75
    # ressortait à ~4,14M au lieu de ~2,07M). On prend directement la
    # ligne "tsage", qui est déjà le total.
    df_pop_totale = (
        df_demo[df_demo["cla_age_5"] == "tsage"][["dept", "Npop"]]
        .rename(columns={"Npop": "pop_totale"})
    )

    # Seniors : 65 ans et plus
    df_seniors = (
        df_demo[df_demo["cla_age_5"].str[:2].isin(
            ["65", "70", "75", "80", "85", "90", "95"]
        )]
        .groupby("dept")["Npop"].sum()
        .reset_index()
        .rename(columns={"Npop": "nb_seniors"})
    )

    # Jeunes : moins de 15 ans
    df_jeunes = (
        df_demo[df_demo["cla_age_5"].str[:2].isin(["00", "05", "10"])]
        .groupby("dept")["Npop"].sum()
        .reset_index()
        .rename(columns={"Npop": "nb_jeunes"})
    )

    # Fusion et calcul des parts
    df_demographie = df_pop_totale.copy()
    df_demographie = df_demographie.merge(df_seniors, on="dept", how="left")
    df_demographie = df_demographie.merge(df_jeunes,  on="dept", how="left")
    df_demographie = df_demographie.merge(df_resp,    on="dept", how="left")
    df_demographie["part_seniors"] = (
        df_demographie["nb_seniors"] / df_demographie["pop_totale"]
    ).round(4)
    df_demographie["part_jeunes"] = (
        df_demographie["nb_jeunes"] / df_demographie["pop_totale"]
    ).round(4)
    df_demographie = df_demographie[["dept", "pop_totale", "part_seniors",
                                     "part_jeunes", "prev_resp_chronique"]]

    print(f"  ✅ Démographie : {len(df_demographie)} départements")

    # ══════════════════════════════════════════════════════════════════
    # SOURCE 2 : INSEE urbanisation
    # ══════════════════════════════════════════════════════════════════
    # Le fichier Excel a une structure complexe avec des lignes de
    # métadonnées en haut. On cherche la ligne contenant les codes
    # département pour trouver le bon skiprows.

    print("\nChargement INSEE urbanisation...")
    df_urban = pd.read_excel(
        geo_pop_dir / "insee_urbanisation.xlsx",
        sheet_name=1,      # 2ème feuille
        header=None,
        skiprows=2,        # sauter les 2 lignes de métadonnées
        usecols=[0, 2]     # colonne 0=dept, colonne 2=taux
        )
    
    df_urban.columns = ["dept", "tx_urbain"]
    df_urban["dept"] = (df_urban["dept"].astype(str).str.strip().str.zfill(2).str.upper())
    df_urban["tx_urbain"] = pd.to_numeric(df_urban["tx_urbain"], errors="coerce").round(1)
    df_urban = df_urban[df_urban["dept"].isin(DEPTS)].dropna()
    print(f"  ✅ Urbanisation : {len(df_urban)} départements")

    # ══════════════════════════════════════════════════════════════════
    # SOURCE 3 : Ameli — densité professionnels de santé
    # ══════════════════════════════════════════════════════════════════
    # 5 fichiers annuels (2020-2024), chacun avec plusieurs feuilles.
    # On prend la feuille "Généralistes et MEP" pour les médecins
    # généralistes et "Spécialistes" pour les spécialistes.
    #
    # Structure de chaque feuille :
    #   col0 → type de spécialité (ex: "01- Médecine générale")
    #   col1 → département (ex: "01- Ain")
    #   col2 → effectif
    #   col3 → population
    #   col4 → densité /100 000 hab  ← on prend ça

    print("\nChargement Ameli...")

    def charger_ameli_feuille(fpath: Path, feuille: str,
                              filtre_type: str) -> pd.DataFrame:
        """
        Charge une feuille d'un fichier Ameli et retourne la densité
        pour un type de professionnel donné, par département.
        """
        df = pd.read_excel(fpath, sheet_name=feuille, header=0)
        df.columns = ["type_ps", "dept_raw", "effectif",
                      "population", "densite"] + list(df.columns[5:])

        # Filtrer sur le type de professionnel voulu
        df = df[df["type_ps"].astype(str).str.contains(
            filtre_type, case=False, na=False
        )].copy()

        # Extraire le code département (2 premiers chiffres)
        df["dept"] = (
            df["dept_raw"].astype(str)
            .str.extract(r"^(\d{2}|2[AB])", expand=False)
            .str.zfill(2)
        )
        df["densite"] = pd.to_numeric(df["densite"], errors="coerce")
        df = df[df["dept"].isin(DEPTS)]

        return df.groupby("dept")["densite"].mean().reset_index()

    # Charger les 5 années et faire la moyenne
    annees_ameli = range(2020, 2025)
    dfs_gen = []
    dfs_spe = []

    for annee in annees_ameli:
        fpath = geo_pop_dir / f"ameli_acces_soins_{annee}.xls"
        if not fpath.exists():
            print(f"  ⚠️  Manquant : {fpath.name}")
            continue
        try:
            df_gen = charger_ameli_feuille(
                fpath, "Généralistes et MEP", "Médecine générale"
            )
            df_gen["annee"] = annee
            dfs_gen.append(df_gen)

            df_spe = charger_ameli_feuille(
                fpath, "Spécialistes", "Médecine"
            )
            df_spe["annee"] = annee
            dfs_spe.append(df_spe)
            print(f"  ✅ Ameli {annee} chargé")
        except Exception as e:
            print(f"  ⚠️  Erreur {annee} : {e}")

    # Moyenne sur toutes les années disponibles
    if dfs_gen:
        df_ameli_gen = (
            pd.concat(dfs_gen)
            .groupby("dept")["densite"]
            .mean()
            .round(2)
            .reset_index()
            .rename(columns={"densite": "densite_med_gen"})
        )
    else:
        df_ameli_gen = pd.DataFrame(columns=["dept", "densite_med_gen"])

    if dfs_spe:
        df_ameli_spe = (
            pd.concat(dfs_spe)
            .groupby("dept")["densite"]
            .mean()
            .round(2)
            .reset_index()
            .rename(columns={"densite": "densite_spe"})
        )
    else:
        df_ameli_spe = pd.DataFrame(columns=["dept", "densite_spe"])

    print(f"  ✅ Ameli généralistes : {len(df_ameli_gen)} depts")
    print(f"  ✅ Ameli spécialistes : {len(df_ameli_spe)} depts")

    # ══════════════════════════════════════════════════════════════════
    # FUSION FINALE
    # ══════════════════════════════════════════════════════════════════
    # On part de la liste complète des départements et on joint
    # chaque source avec how="left" pour conserver tous les depts
    # même si une source est incomplète

    print("\nFusion des 3 sources...")
    dim_geo_pop = pd.DataFrame({"dept": DEPTS})
    dim_geo_pop = dim_geo_pop.merge(df_demographie, on="dept", how="left")
    dim_geo_pop = dim_geo_pop.merge(df_urban,       on="dept", how="left")
    dim_geo_pop = dim_geo_pop.merge(df_ameli_gen,   on="dept", how="left")
    dim_geo_pop = dim_geo_pop.merge(df_ameli_spe,   on="dept", how="left")

    # Rapport de couverture
    print("\n📊 Couverture :")
    for col in dim_geo_pop.columns[1:]:
        pct = dim_geo_pop[col].notna().mean() * 100
        barre = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        statut = "✅" if pct > 80 else "⚠️ "
        print(f"  {statut} {col:<25} {barre} {pct:.0f}%")

    print(f"\n✅ dim_geo_pop : {dim_geo_pop.shape[0]} depts × "
          f"{dim_geo_pop.shape[1]} colonnes")
    return dim_geo_pop


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
dim_geo_pop = build_dim_geo_pop()

if not dim_geo_pop.empty:
    dim_geo_pop.to_parquet(TABLES_DIR / "dim_geo_pop.parquet", index=False)
    print(f"\n✅ Sauvegardé → data/processed/dim_geo_pop.parquet")
    display(dim_geo_pop.head(10))

---
## 5. 💼 `dim_csp` — Catégories Socio-Professionnelles

**Source :** INSEE — Structure de la population active (15-64 ans) par catégorie
socioprofessionnelle, données du recensement **2022**
**Fichier :** `data/raw/insee_csp/insee_csp.xlsx` (feuille `"DEP"`)
**URL :** https://www.insee.fr/fr/statistiques/2012721#tableau-TCRD_014_tab1_departements

Table statique, **1 ligne par département** (comme `dim_geo_pop`, pas de dimension
temporelle) : la structure socioprofessionnelle d'un département évolue trop
lentement pour justifier un suivi mensuel sur la période 2020-2025.

### Variables produites
7 catégories socioprofessionnelles (nomenclature PCS de l'INSEE), en part de la
population active de 15 à 64 ans, exprimées en décimales (ex : `0.163` = 16,3 %) :

| Colonne | Champ source (colonne du fichier Excel) |
|---|---|
| `tx_agriculteurs` | Part des agriculteurs exploitants |
| `tx_artisans` | Part des artisans, commerçants, chefs d'entreprise |
| `tx_cadres` | Part des cadres, professions intellectuelles supérieures |
| `tx_prof_interm` | Part des professions intermédiaires |
| `tx_employes` | Part des employés |
| `tx_ouvriers` | Part des ouvriers |
| `tx_autres` | Autres (sans activité professionnelle identifiée, etc.) |

> **Pourquoi cette variable ?** La catégorie socioprofessionnelle est corrélée à des
> facteurs qui influencent indirectement le recours aux soins d'urgence : exposition
> professionnelle (agriculteurs/ouvriers → pollens, poussières, allergènes),
> couverture complémentaire santé, éloignement des structures de soins (zones
> rurales vs urbaines), habitudes de recours au système de santé. C'est une
> variable de contrôle socio-économique, pas un facteur causal direct des
> pathologies étudiées.


In [ ]:
def build_dim_csp() -> pd.DataFrame:
    """
    Construit la table dim_csp à partir du fichier INSEE CSP (feuille "DEP").

    CLÉ PRIMAIRE : dept
    COLONNES : tx_agriculteurs, tx_artisans, tx_cadres, tx_prof_interm,
               tx_employes, tx_ouvriers, tx_autres  (en décimales, ex: 0.12)
    """
    fpath = RAW_DIR / "insee_csp" / "insee_csp.xlsx"
    if not fpath.exists():
        print(f"⚠️  Fichier manquant : {fpath}")
        return pd.DataFrame()

    df = pd.read_excel(fpath, sheet_name="DEP", header=None, skiprows=4)
    df.columns = ["dept", "nom", "tx_agriculteurs", "tx_artisans",
                  "tx_cadres", "tx_prof_interm", "tx_employes",
                  "tx_ouvriers", "tx_autres"]

    df["dept"] = df["dept"].astype(str).str.strip().str.zfill(2).str.upper()
    df = df[df["dept"].isin(DEPTS)]

    for col in df.columns[2:]:
        df[col] = pd.to_numeric(df[col], errors="coerce") / 100

    cols_out = ["dept"] + list(df.columns[2:])
    df = df[cols_out].reset_index(drop=True)

    print(f"✅ dim_csp : {len(df)} départements × {df.shape[1]} colonnes")
    return df


dim_csp = build_dim_csp()

if not dim_csp.empty:
    dim_csp.to_parquet(TABLES_DIR / "dim_csp.parquet", index=False)
    print(f"✅ Sauvegardé → data/processed/dim_csp.parquet")
    display(dim_csp.head(5))


---
## 6A. 💨 `dim_qualite_air` — Qualité de l'air (AASQA / LCSQA)

Table **dept × annee_mois** (comme `dim_meteo`), construite à partir de 2 sources
complémentaires sur la même page data.gouv.fr :

| Fichier | Rôle | Contenu | Période |
|---|---|---|---|
| `AASQA/FR_E2_YYYY-MM-01.csv` | Mesures | Concentrations horaires par station et polluant | 2021-2025 (1 fichier/mois, pas de données avant 2021) |
| `AASQA/stations_metadata.xls` | Métadonnées | Localisation (lat/lon) de chaque station — feuille `AirQualityStations` | Photo actuelle du réseau |

**Source :** LCSQA (Laboratoire Central de Surveillance de la Qualité de l'Air) — agrège les mesures des réseaux régionaux AASQA
**URL :** https://www.data.gouv.fr/datasets/donnees-temps-reel-de-mesure-des-concentrations-de-polluants-atmospheriques-reglementes-1

> **Pourquoi c'est critique :** O₃ et PM2.5 déclenchent directement les crises d'asthme. NO₂ potentialise les réponses allergiques. La pollution particulaire fragilise les voies respiratoires des nourrissons (bronchiolite).

### Variables produites
| Colonne | Polluant source | Signification |
|---|---|---|
| `no_moy` | NO (monoxyde d'azote) | Concentration mensuelle moyenne (µg/m³) |
| `no2_moy` | NO2 (dioxyde d'azote) | Concentration mensuelle moyenne — facteur allergie + asthme |
| `o3_moy` | O3 (ozone) | Concentration mensuelle moyenne — déclencheur crises d'asthme |
| `pm10_moy` | PM10 (particules < 10 µm) | Concentration mensuelle moyenne — facteur asthme & bronchiolite |
| `pm25_moy` | PM2.5 (particules < 2,5 µm) | Concentration mensuelle moyenne — facteur asthme sévère |
| `so2_moy` | SO2 (dioxyde de soufre) | Concentration mensuelle moyenne (peu de stations, cf. limites) |
| `nb_jours_pm10_eleve_est` | PM10 | Estimation du nb de jours/mois où PM10 > 50 µg/m³ (seuil OMS), à partir de la proportion de mesures horaires dépassant le seuil × 30 |

Seules les mesures avec `validité == 1` (mesure valide) sont conservées ; les
valeurs négatives ou > 1000 (aberrantes) sont mises à NaN avant agrégation.

### ⚠️ Bug corrigé : extraction du département depuis le code site
La version précédente du pipeline supposait que le `code site` AASQA suivait le
format `FR + 2 chiffres dept + 3 chiffres station` (ex : `FR01005 → département 01`).
**Cette hypothèse est fausse** : le code site est un numéro d'ordre interne au réseau
de mesure, pas un code géographique. Exemples réels :

```
code site = FR01011, nom site = "Metz-Centre"    → en réalité département 57 (Moselle), pas 01 (Ain)
code site = FR02001, nom site = "Berre l'Étang"  → en réalité département 13 (Bouches-du-Rhône), pas 02 (Aisne)
```

Conséquence : les valeurs étaient attribuées à des départements complètement erronés
pour une grande partie des stations, et Paris (75) — pourtant fortement instrumenté —
ressortait sans aucune donnée (aucun code site ne commence par "75").

**Correction :** comme pour les stations SYNOP/MENSQ (météo), on utilise les
coordonnées lat/lon réelles des stations (fournies dans `stations_metadata.xls`)
et un reverse geocoding pour retrouver le vrai département (`build_station_dict_aasqa()`,
résultat mis en cache dans `stations_aasqa_dept.parquet`).

### Couverture obtenue (après correction)
| | Avant (bug) | Après (correction) |
|---|---|---|
| Départements couverts | 42 / 96 | **92 / 96** |
| `no_moy` / `no2_moy` / `pm10_moy` | ~36% | **97-98%** |
| `pm25_moy` | ~36% | **91%** |
| `o3_moy` | ~32% | **87%** |
| `so2_moy` | ~20% | 28% (limite réseau réelle, pas un bug — cf. ci-dessous) |

### Limites restantes (réelles, pas des bugs)
- **2020 absent** : le flux temps réel LCSQA n'est archivé qu'à partir de 2021.
- **SO₂ ~28%** : peu de stations mesurent ce polluant à l'échelle nationale — c'est
  une limite du réseau de surveillance, pas un problème d'extraction.
- **4 départements toujours non couverts : 09 (Ariège), 11 (Aude), 46 (Lot), 48 (Lozère)**
  - **11, 46, 48** : aucune station recensée dans `stations_metadata.xls` sur ces
    départements (zéro station, même après reverse geocoding).
  - **09** : 1 station recensée, mais aucune mesure exploitable dans les fichiers
    `FR_E2_*.csv` (ne mesure aucun des 6 polluants suivis, ou toutes ses valeurs
    échouent le filtre `validité == 1`).

  Ce sont 4 départements ruraux du sud-ouest (Occitanie), peu peuplés et peu
  industrialisés — cohérent avec un réseau de surveillance dimensionné en priorité
  sur les zones à enjeux (agglomérations, axes routiers, sites industriels). C'est
  un vrai vide du réseau de mesure, pas un artefact de l'extraction.


In [35]:
df = pd.read_csv(
    "data/raw/AASQA/FR_E2_2021-01-01.csv",
    sep=";", nrows=5
)
print("=== FR_E2 (mesures) ===")
print(df.columns.tolist())
display(df.head())

# Métadonnées des stations (pour la correspondance station -> département)
df_stations_check = pd.read_excel(
    "data/raw/AASQA/stations_metadata.xls",
    sheet_name="AirQualityStations", nrows=5
)
print("\n=== stations_metadata (AirQualityStations) ===")
print(df_stations_check.columns.tolist())
display(df_stations_check[["NatlStationCode", "Name", "Municipality", "Latitude", "Longitude"]])

=== FR_E2 (mesures) ===
['Date de début', 'Date de fin', 'Organisme', 'code zas', 'Zas', 'code site', 'nom site', "type d'implantation", 'Polluant', "type d'influence", 'discriminant', 'Réglementaire', "type d'évaluation", 'procédure de mesure', 'type de valeur', 'valeur', 'valeur brute', 'unité de mesure', 'taux de saisie', 'couverture temporelle', 'couverture de données', 'code qualité', 'validité']


,Date de début,Date de fin,Organisme,code zas,Zas,code site,nom site,type d'implantation,Polluant,type d'influence,...,procédure de mesure,type de valeur,valeur,valeur brute,unité de mesure,taux de saisie,couverture temporelle,couverture de données,code qualité,validité
0,2021/01/01 00:00:00,2021/01/01 01:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,18.9,18.875,µg-m3,NaN,NaN,NaN,A,1
1,2021/01/01 01:00:00,2021/01/01 02:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,10.8,10.800,µg-m3,NaN,NaN,NaN,A,1
2,2021/01/01 02:00:00,2021/01/01 03:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,10.3,10.300,µg-m3,NaN,NaN,NaN,A,1
3,2021/01/01 03:00:00,2021/01/01 04:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,6.4,6.400,µg-m3,NaN,NaN,NaN,A,1
4,2021/01/01 04:00:00,2021/01/01 05:00:00,ATMO GRAND EST,FR44ZAG02,ZAG METZ,FR01005,Hayange,Périurbaine,PM10,Industrielle,...,Auto PM_Conf_app MP101M-RST,moyenne horaire validée,8.1,8.050,µg-m3,NaN,NaN,NaN,A,1



=== stations_metadata (AirQualityStations) ===
['GMLID', 'LocalId', 'Namespace', 'Version', 'NatlStationCode', 'Name', 'Municipality', 'EUStationCode', 'ActivityBegin', 'ActivityEnd', 'Latitude', 'Longitude', 'SRSName', 'Altitude', 'AltitudeUnit', 'AreaClassification', 'BelongsTo']


,NatlStationCode,Name,Municipality,Latitude,Longitude
0,FR19012,Brest Mace,BREST,48.386180,-4.486600
1,FR34051,Chateauroux Sud,CHÂTEAUROUX,46.798280,1.693139
2,FR12047,Bessières-ECONOTRE,BESSIÈRES,43.803140,1.595475
3,FR16034,Strasbourg Clemenceau,STRASBOURG,48.590430,7.744983
4,FR30026,Luneville,LUNÉVILLE,48.584824,6.483900


In [ ]:
import re

def build_station_dict_aasqa() -> dict:
    """
    Construit le dictionnaire {NatlStationCode → code_dept} à partir des
    métadonnées officielles des stations AASQA (feuille "AirQualityStations"
    de stations_metadata.xls) et d'un reverse geocoding (lat/lon → département).

    ⚠️ Remplace l'ancienne fonction _extract_dept_from_site(), qui supposait à
    tort que le "code site" encodait le département dans ses 2 premiers
    chiffres (ex: "FR01005" → dept 01). En réalité ce code est un numéro
    d'ordre interne au réseau de mesure, pas un code géographique (cf. note
    dans la cellule markdown ci-dessus). Même logique que pour les stations
    SYNOP/MENSQ (météo) : on repart des coordonnées réelles.

    Sauvegarde aussi data/processed/stations_aasqa_dept.parquet pour réutilisation.

    Retourne : dict { NatlStationCode (str) → code_dept (str) }
    """
    stations_path = RAW_DIR / "AASQA" / "stations_metadata.xls"
    if not stations_path.exists():
        print(f"⚠️  Métadonnées stations introuvables : {stations_path}")
        return {}

    df_stations = pd.read_excel(stations_path, sheet_name="AirQualityStations")
    print(f"Stations totales : {len(df_stations)}")

    # Reverse geocoding : lat/lon → département
    coords = list(zip(df_stations["Latitude"], df_stations["Longitude"]))
    resultats = rg.search(coords)
    df_stations["dept_nom"] = [r["admin2"] for r in resultats]
    df_stations["pays"]     = [r["cc"]     for r in resultats]

    # Garder uniquement France métropolitaine
    df_fr = df_stations[df_stations["pays"] == "FR"].copy()
    print(f"Stations France  : {len(df_fr)}")

    # Nom département (reverse geocoder) → code INSEE
    DEPT_NOM_TO_CODE = {
        "Departement de l'Ain": "01", "Departement de l'Aisne": "02",
        "Departement de l'Allier": "03", "Departement des Alpes-de-Haute-Provence": "04",
        "Departement des Hautes-Alpes": "05",
        "Departement des Alpes-Maritimes": "06", "Departement de l'Ardeche": "07",
        "Departement des Ardennes": "08", "Departement de l'Ariege": "09",
        "Departement de l'Aube": "10", "Departement de l'Aude": "11",
        "Departement de l'Aveyron": "12", "Departement des Bouches-du-Rhone": "13",
        "Departement du Calvados": "14", "Departement du Cantal": "15",
        "Departement de la Charente": "16", "Departement de la Charente-Maritime": "17",
        "Departement du Cher": "18", "Departement de la Correze": "19",
        "Departement de la Cote-d'Or": "21", "Departement des Cotes-d'Armor": "22",
        "Departement de la Creuse": "23", "Departement de la Dordogne": "24",
        "Departement du Doubs": "25", "Departement de la Drome": "26",
        "Departement de l'Eure": "27", "Departement d'Eure-et-Loir": "28",
        "Departement du Finistere": "29", "Departement du Gard": "30",
        "Departement de la Haute-Garonne": "31", "Departement du Gers": "32",
        "Departement de la Gironde": "33", "Departement de l'Herault": "34",
        "Departement d'Ille-et-Vilaine": "35", "Departement de l'Indre": "36",
        "Departement d'Indre-et-Loire": "37", "Departement de l'Isere": "38",
        "Departement du Jura": "39", "Departement des Landes": "40",
        "Departement du Loir-et-Cher": "41", "Departement de la Loire": "42",
        "Departement de la Haute-Loire": "43", "Departement de la Loire-Atlantique": "44",
        "Departement du Loiret": "45", "Departement du Lot": "46",
        "Departement du Lot-et-Garonne": "47", "Departement de la Lozere": "48",
        "Departement du Maine-et-Loire": "49", "Departement de la Manche": "50",
        "Departement de la Marne": "51", "Departement de la Haute-Marne": "52",
        "Departement de la Mayenne": "53", "Departement de Meurthe-et-Moselle": "54",
        "Departement de la Meuse": "55", "Departement du Morbihan": "56",
        "Departement de la Moselle": "57", "Departement de la Nievre": "58",
        "Departement du Nord": "59", "Departement de l'Oise": "60",
        "Departement de l'Orne": "61", "Departement du Pas-de-Calais": "62",
        "Departement du Puy-de-Dome": "63", "Departement des Pyrenees-Atlantiques": "64",
        "Departement des Hautes-Pyrenees": "65", "Departement des Pyrenees-Orientales": "66",
        "Departement du Bas-Rhin": "67", "Departement du Haut-Rhin": "68",
        "Departement du Rhone": "69", "Departement de la Haute-Saone": "70",
        "Departement de Saone-et-Loire": "71", "Departement de la Sarthe": "72",
        "Departement de la Savoie": "73", "Departement de la Haute-Savoie": "74",
        "Departement de Paris": "75", "Departement de la Seine-Maritime": "76",
        "Departement de Seine-et-Marne": "77", "Departement des Yvelines": "78",
        "Departement des Deux-Sevres": "79", "Departement de la Somme": "80",
        "Departement du Tarn": "81", "Departement du Tarn-et-Garonne": "82",
        "Departement du Var": "83", "Departement du Vaucluse": "84",
        "Departement de la Vendee": "85", "Departement de la Vienne": "86",
        "Departement de la Haute-Vienne": "87", "Departement des Vosges": "88",
        "Departement de l'Yonne": "89", "Departement du Territoire de Belfort": "90",
        "Departement de l'Essonne": "91", "Departement des Hauts-de-Seine": "92",
        "Departement de la Seine-Saint-Denis": "93", "Departement du Val-de-Marne": "94",
        "Departement du Val-d'Oise": "95",
        "Departement de la Corse-du-Sud": "2A", "Departement de la Haute-Corse": "2B",
        # Variantes orthographiques rencontrées dans le gazetteer de reverse_geocoder
        # (mêmes départements, formulation différente selon l'entrée géographique) :
        "Paris": "75",
        "Departement de Seine-Saint-Denis": "93",
        "Departement de Maine-et-Loire": "49",
        "Territoire de Belfort": "90",
    }

    df_fr["dept"] = df_fr["dept_nom"].map(DEPT_NOM_TO_CODE)

    # Diagnostics
    sans_code = df_fr[df_fr["dept"].isna()][["NatlStationCode", "Name", "dept_nom"]]
    if len(sans_code):
        print(f"⚠️  {len(sans_code)} stations sans correspondance département :")
        print(sans_code.to_string())
    else:
        print("✅ Toutes les stations mappées")

    df_fr = df_fr.dropna(subset=["dept"])

    # Sauvegarder pour les runs futurs
    df_fr[["NatlStationCode", "Name", "dept"]].to_parquet(
        TABLES_DIR / "stations_aasqa_dept.parquet", index=False
    )
    print(f"✅ Sauvegardé → data/processed/stations_aasqa_dept.parquet ({len(df_fr)} stations)")
    print(f"✅ Départements couverts : {df_fr['dept'].nunique()} / {len(DEPTS)}")

    return dict(zip(df_fr["NatlStationCode"], df_fr["dept"]))


def build_dim_qualite_air(station_to_dept: dict) -> pd.DataFrame:
    """
    Construit la table dim_qualite_air à partir des 60 fichiers AASQA mensuels.

    SOURCE : LCSQA — Concentrations de polluants atmosphériques réglementés
    Fichiers : FR_E2_YYYY-MM-01.csv (un par mois, 2021-2025)

    EXTRACTION DU DÉPARTEMENT :
    ───────────────────────────
    Via station_to_dept (cf. build_station_dict_aasqa() ci-dessus), construit
    par reverse geocoding des coordonnées réelles des stations — PAS depuis le
    code site, qui n'encode pas le département (cf. note markdown ci-dessus).

    POLLUANTS TRAITÉS :
    ───────────────────
    PM10  → particules fines (µg/m³) — facteur asthme & bronchiolite
    PM2.5 → particules très fines     — facteur asthme sévère
    NO2   → dioxyde d'azote           — facteur allergie + asthme
    O3    → ozone                     — déclencheur crises d'asthme

    COLONNES PRODUITES :
    ────────────────────
    dept          → code département
    annee_mois    → période (ex: "2021-03")
    pm10_moy      → concentration mensuelle moyenne PM10 (µg/m³)
    pm25_moy      → concentration mensuelle moyenne PM2.5
    no2_moy       → concentration mensuelle moyenne NO2
    o3_moy        → concentration mensuelle moyenne O3
    nb_jours_pm10_eleve_est → estimation du nb de jours/mois avec PM10 > 50 µg/m³ (seuil OMS)

    CLÉ PRIMAIRE : dept × annee_mois
    """

    aasqa_dir = RAW_DIR / "AASQA"
    if not aasqa_dir.exists():
        print(f"⚠️  Dossier manquant : {aasqa_dir}")
        return pd.DataFrame()

    fichiers = sorted(aasqa_dir.glob("FR_E2_*.csv"))
    if not fichiers:
        print("⚠️  Aucun fichier FR_E2_*.csv trouvé")
        return pd.DataFrame()

    print(f"Chargement de {len(fichiers)} fichiers AASQA...")

    # Polluants d'intérêt et leurs noms normalisés
    POLLUANTS = {
        "PM10":  "pm10",
        "PM2.5": "pm25",
        "NO2":   "no2",
        "O3":    "o3",
        "NO":    "no",    # optionnel
        "SO2":   "so2",   # optionnel
    }

    dfs = []

    for fpath in fichiers:

        # Extraire l'année et le mois depuis le nom de fichier
        # Format : FR_E2_2021-01-01.csv → annee_mois = "2021-01"
        match = re.search(r'FR_E2_(\d{4}-\d{2})-\d{2}\.csv', fpath.name)
        if not match:
            continue
        annee_mois = match.group(1)

        try:
            df = pd.read_csv(
                fpath,
                sep=";",
                encoding="utf-8",
                low_memory=False,
                dtype=str          # tout en texte d'abord
            )

            # ── Colonnes utiles ──────────────────────────────────────────────
            # Renommer pour standardiser
            rename = {
                "Date de début":    "date_debut",
                "code site":        "code_site",
                "Polluant":         "polluant",
                "valeur":           "valeur",
                "validité":         "validite",
                "unité de mesure":  "unite",
            }
            df = df.rename(columns={k: v for k, v in rename.items()
                                    if k in df.columns})

            df["dept"] = df["code_site"].map(station_to_dept)
            df = df.dropna(subset=["dept"])
            df = df[df["dept"].isin(DEPTS)]

            # ── Filtrage sur les polluants d'intérêt ─────────────────────
            df = df[df["polluant"].isin(POLLUANTS.keys())].copy()
            if df.empty:
                continue

            # ── Filtrage sur données valides (validite = 1) ───────────────
            df["validite"] = pd.to_numeric(df["validite"], errors="coerce")
            df = df[df["validite"] == 1]

            # ── Conversion numérique de la valeur ─────────────────────────
            df["valeur"] = pd.to_numeric(df["valeur"], errors="coerce")
            df = df.dropna(subset=["valeur"])

            # ── Nettoyage des valeurs aberrantes ──────────────────────────
            # Valeurs négatives ou extrêmes → NaN
            df.loc[df["valeur"] < 0, "valeur"] = np.nan
            df.loc[df["valeur"] > 1000, "valeur"] = np.nan

            df["annee_mois"] = annee_mois

            # ── Normaliser le nom du polluant ──────────────────────────────
            df["polluant_norm"] = df["polluant"].map(POLLUANTS)

            dfs.append(df[["dept", "annee_mois", "polluant_norm",
                            "valeur"]].copy())

        except Exception as e:
            print(f"  ⚠️  Erreur {fpath.name} : {e}")

    if not dfs:
        print("❌ Aucune donnée chargée")
        return pd.DataFrame()

    df_all = pd.concat(dfs, ignore_index=True)
    print(f"Total lignes valides : {len(df_all):,}")
    print(f"Polluants disponibles : {df_all['polluant_norm'].unique().tolist()}")

    # ── Agrégation mensuelle par département × polluant ───────────────────
    # Chaque fichier contient déjà un seul mois → groupby dept × polluant
    df_pivot = (
        df_all.groupby(["dept", "annee_mois", "polluant_norm"])["valeur"]
        .mean()
        .round(2)
        .reset_index()
    )

    # Pivot : une colonne par polluant
    df_wide = df_pivot.pivot_table(
        index=["dept", "annee_mois"],
        columns="polluant_norm",
        values="valeur",
        aggfunc="mean"
    ).reset_index()

    # Renommer les colonnes
    df_wide.columns.name = None
    rename_cols = {p: f"{p}_moy" for p in POLLUANTS.values()
                   if p in df_wide.columns}
    df_wide = df_wide.rename(columns=rename_cols)

    # ── Indicateur jours PM10 élevé ──────────────────────────────────────
    # Compte le nombre de jours calendaires distincts où PM10 > 50 µg/m³
    if "pm10" in df_all["polluant_norm"].values:
        df_pm10 = df_all[df_all["polluant_norm"] == "pm10"].copy()
        # Reconstruire la date réelle depuis date_debut si disponible,
        # sinon on sait que chaque fichier = 1 mois → on approche par mesure unique
        df_pm10["jour_eleve"] = (df_pm10["valeur"] > 50).astype(int)
        # Moyenne par (dept, annee_mois) : proportion de mesures dépassant le seuil
        # (proxy du nb de jours, car les mesures sont ~horaires)
        df_jours = (
            df_pm10.groupby(["dept", "annee_mois"])["jour_eleve"]
            .mean()
            .mul(30)         # × 30 jours → estimation du nb de jours/mois
            .round(1)
            .reset_index()
            .rename(columns={"jour_eleve": "nb_jours_pm10_eleve_est"})
        )
        df_wide = df_wide.merge(df_jours, on=["dept", "annee_mois"],
                                how="left")

    print(f"\n✅ dim_qualite_air : {df_wide.shape[0]:,} lignes × "
          f"{df_wide.shape[1]} colonnes")
    print(f"Période      : {df_wide['annee_mois'].min()} → "
          f"{df_wide['annee_mois'].max()}")
    print(f"Départements : {df_wide['dept'].nunique()} couverts")
    print(f"Colonnes     : {list(df_wide.columns)}")

    # Couverture par polluant
    print("\n📊 Couverture par polluant :")
    for col in [c for c in df_wide.columns if c.endswith("_moy")]:
        pct   = df_wide[col].notna().mean() * 100
        barre = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        print(f"  {col:<20} {barre} {pct:.0f}%")

    return df_wide


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
print("Construction du dictionnaire station -> dept (AASQA)...")
station_to_dept_aasqa = build_station_dict_aasqa()

print("\nConstruction de dim_qualite_air...")
dim_qualite_air = build_dim_qualite_air(station_to_dept_aasqa)

if not dim_qualite_air.empty:
    dim_qualite_air.to_parquet(
        TABLES_DIR / "dim_qualite_air.parquet", index=False
    )
    print(f"\n✅ Sauvegardé → data/processed/dim_qualite_air.parquet")
    display(dim_qualite_air.head(10))

---
## 6B. 🌿 `dim_pollen` — Concentration pollinique (RNSA)

**Source :** Réseau National de Surveillance Aérobiologique (RNSA)
**URL :** https://www.data.gouv.fr/datasets/donnees-historiques-de-surveillance-des-pollens-et-des-moisissures

### Fichier à télécharger → `data/raw/BDD_daily_2020_2025/`
> Ce dossier rassemble les mesures effectuées sur 133 capteurs entre 1987 et 2024 et compilées par le RNSA. Chaque fichier contient les données d'une année et d'un capteur. Les données sont ici présentées par jour et par taxons.

> ⚠️ **Le RNSA a été placé en liquidation judiciaire le 26/03/2025** (rapport
> d'inspection administrative pointant une gestion jugée opaque et une mission
> "très mal" remplie). La surveillance pollinique est reprise par Atmo France,
> qui a lancé le 02/04/2025 un nouvel "indice pollen" — **mais celui-ci n'est
> pas utilisable pour ce projet** : historisation seulement depuis le
> 06/01/2026 (rien sur 2020-2025), granularité communale, et il s'agit d'un
> **indice catégoriel** (très faible/faible/modéré/élevé/très élevé/extrêmement
> élevé) et non d'une concentration continue (grains/m³) — pas le même type de
> variable que `pollen_*_moy`. Conclusion : le RNSA (bien que défunt) reste la
> seule source disponible pour l'historique 2020-2025 ; les fichiers déjà
> téléchargés sont donc conservés tels quels.

### ➕ Variables ajoutées : moisissures (Alternaria, Cladosporium)
Les mêmes fichiers RNSA contiennent aussi des colonnes de spores fongiques
(moisissures), en plus des pollens — 32 taxons fongiques présents dans le
schéma des fichiers, mais tous les capteurs ne les analysent pas (ex : Lyon et
Strasbourg mesurent quasiment tout le panel fongique, Toulouse et
Clermont-Ferrand ne mesurent presque aucune moisissure). On extrait les 2 taxons
les mieux documentés comme facteurs d'asthme/allergie sévère et les mieux
couverts par le réseau :

| Colonne | Taxon | Couverture stations (≥1 valeur sur 93 villes) |
|---|---|---|
| `moisissure_alternaria_moy` | Alternaria | 66/93 (71%) |
| `moisissure_cladosporium_moy` | Cladosporium | 66/93 (71%) |

Au niveau dept × mois dans `dim_pollen`, la couverture réelle tombe à ~26%
(même limite de fond que le reste de `dim_pollen` : seuls 49/96 départements
ont une ville RNSA mappée, cf. limite ci-dessous). Extraction ajoutée via
`MOISISSURE_CLES`, même logique que `POLLEN_CLES` mais colonnes distinctes
(non incluses dans `pollen_global_max`/`nb_jours_eleve`, qui restent spécifiques
au pollen).

### ⚠️ Limite connue : ~40% des capteurs silencieusement ignorés
`VILLE_TO_DEPT` associe manuellement le préfixe de ville dans le nom de fichier
à un département. Comparaison entre les 93 préfixes de ville réellement présents
dans `data/raw/BDD_daily_2020_2025/` et les clés de `VILLE_TO_DEPT` :

- **37 préfixes de ville présents dans les fichiers mais absents du dictionnaire**
  (ex : `LILLE`, `CLERMONT` — Clermont-Ferrand, tapé `CLERMON` dans le dico,
  `CHARLEVI` — Charleville-Mézières, tapé `CHARLEV`/`CHARLEV2`, `MACON`,
  `NARBONNE`, `NEVERS`, `SAINT-ET` — Saint-Étienne, `NICE2`, `TULLE`, `GAP`,
  `ROANNE`, `CASTRES`, `CHOLET`, `DINAN`, `DOLE`, `MTMARSAN`, `PONTIVY`, etc.)
- Ces lignes sont ignorées **sans aucun avertissement** dans le code
  (`if dept is None: continue`) — le problème n'était visible qu'en comparant
  manuellement fichiers réels et dictionnaire.
- **46 clés du dictionnaire ne correspondent à aucun fichier réel** (probablement
  d'anciennes tentatives de nommage jamais nettoyées).
- Le dictionnaire porte lui-même la trace d'erreurs passées jamais totalement
  résolues (ex : commentaire `"BLETTERA": "07", # ... Bletterans → 39 Jura ?`
  resté en suspens, ou `"SAINTBRI": "22", # ... corrigé depuis 71`).

**Pourquoi ce n'est pas corrigé de la même façon que SYNOP/MENSQ/AASQA :**
contrairement aux stations météo et AASQA, les fichiers RNSA n'embarquent aucune
coordonnée lat/lon — seul le nom de ville est disponible (dans le nom de fichier
et l'en-tête de colonne). Une correction par reverse geocoding n'est donc pas
possible ici. On a aussi vérifié s'il existait, comme pour AASQA (`stations_metadata.xls`),
un référentiel officiel des stations RNSA avec coordonnées : **il n'existe pas**
(le seul fichier `Documentation.xlsx` fourni sur data.gouv.fr est une simple
description textuelle du jeu de données, pas un registre de stations). Corriger
les 37 entrées manquantes demanderait donc une vérification manuelle ville par
ville, décision volontairement reportée pour l'instant (cf. `MEMORY`/décision
projet : coût jugé non prioritaire par rapport au reste du pipeline).

### ⚠️ Bug connu : `nb_jours_eleve` n'est pas un nombre de jours
```python
agg_dict["nb_jours_eleve"] = ("jour_eleve", "max")  # max sur tout le mois × toutes les villes du dept
```
`jour_eleve` est un indicateur binaire (0/1) par jour. Agrégé avec `.max()` sur
tout un mois, le résultat ne peut valoir que 0 ou 1 — confirmé empiriquement sur
`dim_pollen.parquet` (aucune valeur autre que 0/1 sur les 1 686 lignes). Le nom
de colonne et la docstring annoncent "nb de jours avec pollen élevé", mais la
variable réelle est en fait "au moins un jour élevé ce mois-ci (oui/non)" — pas
un décompte. Non corrigé pour l'instant, même décision que ci-dessus.


In [29]:
pollen_dir = Path("data/raw/BDD_daily")

# Filtrer uniquement les fichiers 2020-2025
fichiers_utiles = []
for f in pollen_dir.glob("*.xls"):
    match = re.search(r'(\d{4})-\d{2}-\d{2}_to_', f.name)
    if match:
        annee = int(match.group(1))
        if 2020 <= annee <= 2025:
            fichiers_utiles.append(f)

print(f"Fichiers 2020-2025 : {len(fichiers_utiles)}")
for f in sorted(fichiers_utiles)[:10]:
    print(f"  {f.name}")
    
import shutil

pollen_dir = Path("data/raw/BDD_daily")
dest_dir = Path("data/raw/BDD_daily_2020_2025")
dest_dir.mkdir(parents=True, exist_ok=True)

copiés = 0
for f in pollen_dir.glob("*.xls"):
    match = re.search(r'(\d{4})-\d{2}-\d{2}_to_', f.name)
    if match:
        annee = int(match.group(1))
        if 2020 <= annee <= 2025:
            shutil.copy2(f, dest_dir / f.name)
            copiés += 1

print(f"✅ {copiés} fichiers copiés → {dest_dir}")

Fichiers 2020-2025 : 253
  Particle_Extract_AGEN_2020-01-01_to_2020-12-31.xls
  Particle_Extract_AGEN_2021-01-01_to_2021-12-31.xls
  Particle_Extract_AGEN_2023-01-01_to_2023-12-31.xls
  Particle_Extract_AIXENPRO_2020-01-01_to_2020-12-31.xls
  Particle_Extract_AIXENPRO_2021-01-01_to_2021-12-31.xls
  Particle_Extract_AIXENPRO_2023-01-01_to_2023-12-31.xls
  Particle_Extract_AJACCIO_2020-01-01_to_2020-12-31.xls
  Particle_Extract_AJACCIO_2021-01-01_to_2021-12-31.xls
  Particle_Extract_AJACCIO_2023-01-01_to_2023-12-31.xls
  Particle_Extract_AMBERIEU_2020-01-01_to_2020-12-31.xls
✅ 253 fichiers copiés → data/raw/BDD_daily_2020_2025


In [30]:
pollen_dir = Path("data/raw/BDD_daily_2020_2025")
villes = set()
for f in pollen_dir.glob("*.xls"):
    match = re.search(r'Particle_Extract_(.+?)_\d{4}', f.name)
    if match:
        villes.add(match.group(1))

print(f"Total villes : {len(villes)}")
for v in sorted(villes):
    print(f"  {v}")

Total villes : 93
  AGEN
  AIXENPRO
  AJACCIO
  AMBERIEU
  AMIENS
  ANDORRA
  ANGERS
  ANGOULEM
  ANNECY
  ANNEMASS
  ANTONY
  AURILLAC
  AVIGNON
  BAGNOLS
  BART
  BERRIAS
  BESANCON
  BLETTERA
  BORDPESS
  BOURGENB
  BOURGES
  BOURGOIN
  BREST
  BRUSSLAN
  CAEN
  CASTRES
  CHALON-S
  CHAMBERY
  CHARLEVI
  CHAUMONT
  CHOLET
  CLERMONT
  DIJON
  DINAN
  DOLE
  DRAGUIGN
  GAP
  GENAS
  GLEIZE
  GONESSE
  GRENOBLE
  LAROCHE-
  LAROCHL
  LEMANS
  LEPUYENV
  LILLE
  LIMOGES
  LORIENT
  LURE
  LYON
  MACON
  MAREUIL
  MARSEILL
  METZ
  MONTLUCO
  MONTPELL
  MTMARSAN
  MULHOUSE
  NANCY
  NANTES
  NARBONNE
  NEVERS
  NICE
  NICE2
  NIORT
  ORLEANS
  PARIS
  PERIGUEU
  POITIERS
  PONTIVY
  REIMS
  RENNES
  ROANNE
  ROUEN
  ROUSSILL
  SACLAY
  SACLAYSP
  SAINT-ET
  SAINTDIE
  ST-BRIEU
  STALBAN
  STEFOY
  STRASBOU
  TOULON
  TOULOUSE
  TOULOUSM
  TOURS
  TROYES
  TULLE
  VALDAHON
  VALENCE
  VICHY
  VILLENEU


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TABLE DE CORRESPONDANCE : ville RNSA → code département
# ══════════════════════════════════════════════════════════════════════════════
# Construite manuellement à partir des 93 villes du dataset

VILLE_TO_DEPT = {
    "AGEN":       "47",  # Lot-et-Garonne
    "AIXENPRO":   "13",  # Bouches-du-Rhône (Aix-en-Provence)
    "AJACCIO":    "2A",  # Corse-du-Sud
    "AMBERIEU":   "01",  # Ain (Ambérieu-en-Bugey)
    "AMIENS":     "80",  # Somme
    "ANDORRA":    None,  # Andorre → hors France, ignoré
    "ANGERS":     "49",  # Maine-et-Loire
    "ANGOULEM":   "16",  # Charente (Angoulême)
    "ANNECY":     "74",  # Haute-Savoie
    "ANNEMASS":   "74",  # Haute-Savoie (Annemasse)
    "ANTONY":     "92",  # Hauts-de-Seine
    "AURILLAC":   "15",  # Cantal
    "AVIGNON":    "84",  # Vaucluse
    "BAGNOLS":    "30",  # Gard (Bagnols-sur-Cèze)
    "BART":       "25",  # Doubs
    "BERRIAS":    "07",  # Ardèche
    "BESANCON":   "25",  # Doubs
    "BEZIERS":    "34",  # Hérault
    "BORDEAUX":   "33",  # Gironde
    "BOURGES":    "18",  # Cher
    "BREST":      "29",  # Finistère
    "BRIANCON":   "05",  # Hautes-Alpes
    "CAEN":       "14",  # Calvados
    "CARCASS":    "11",  # Aude (Carcassonne)
    "CHAMBERY":   "73",  # Savoie
    "CHARLEV":    "08",  # Ardennes (Charleville-Mézières)
    "CHARTRES":   "28",  # Eure-et-Loir
    "CHERBOU":    "50",  # Manche (Cherbourg)
    "CLERMON":    "63",  # Puy-de-Dôme (Clermont-Ferrand)
    "COLMAR":     "68",  # Haut-Rhin
    "CREIL":      "60",  # Oise
    "DIJON":      "21",  # Côte-d'Or
    "DIGNE":      "04",  # Alpes-de-Haute-Provence
    "DUNKERQ":    "59",  # Nord (Dunkerque)
    "EMBRUN":     "05",  # Hautes-Alpes
    "GRENOBLE":   "38",  # Isère
    "LAON":       "02",  # Aisne
    "LAROCHE":    "89",  # Yonne (Laroche-Saint-Cydroine / Auxerre)
    "LAVAL":      "53",  # Mayenne
    "LEMANS":     "72",  # Sarthe (Le Mans)
    "LENS":       "62",  # Pas-de-Calais
    "LILE":       "59",  # Nord (Lille)
    "LIMOGES":    "87",  # Haute-Vienne
    "LORIENT":    "56",  # Morbihan
    "LYON":       "69",  # Rhône
    "MARSEILL":   "13",  # Bouches-du-Rhône
    "METZ":       "57",  # Moselle
    "MONTLUEL":   "01",  # Ain
    "MONTPELL":   "34",  # Hérault (Montpellier)
    "MULHOUSE":   "68",  # Haut-Rhin
    "NANCY":      "54",  # Meurthe-et-Moselle
    "NANTES":     "44",  # Loire-Atlantique
    "NICE":       "06",  # Alpes-Maritimes
    "NIMES":      "30",  # Gard
    "NIORT":      "79",  # Deux-Sèvres
    "ORLEANS":    "45",  # Loiret
    "PARIS":      "75",  # Paris
    "PAU":        "64",  # Pyrénées-Atlantiques
    "PERPIGNA":   "66",  # Pyrénées-Orientales (Perpignan)
    "POITIERS":   "86",  # Vienne
    "REIMS":      "51",  # Marne
    "RENNES":     "35",  # Ille-et-Vilaine
    "ROUEN":      "76",  # Seine-Maritime
    "SAINTBRI":   "22",  # Côtes-d'Armor (Saint-Brieuc) — corrigé depuis 71
    "SAINTETIE":  "42",  # Loire (Saint-Étienne)
    "SAINTMAL":   "35",  # Ille-et-Vilaine (Saint-Malo)
    "SAINTQUEN":  "80",  # Somme (Saint-Quentin)
    "SARREBOU":   "57",  # Moselle (Sarreboug)
    "STALBAN":    "81",  # Tarn (Saint-Alban)
    "STEFOY":     "33",  # Gironde (Sainte-Foy-la-Grande)
    "STRASBOU":   "67",  # Bas-Rhin (Strasbourg)
    "TOULON":     "83",  # Var
    "TOULOUSE":   "31",  # Haute-Garonne
    "TOULOUSM":   "31",  # Haute-Garonne (autre station Toulouse)
    "TOURS":      "37",  # Indre-et-Loire
    "TROYES":     "10",  # Aube
    "VALENCE":    "26",  # Drôme
    "VANNES":     "56",  # Morbihan
    "VERSAIL":    "78",  # Yvelines (Versailles)
    "VICHY":      "03",  # Allier
    "VIENNE":     "38",  # Isère (Vienne)
    "VILLENEU":   "47",  # Lot-et-Garonne (Villeneuve-sur-Lot)
    "ST-BRIEU":   "22",  # Côtes-d'Armor (Saint-Brieuc)
    "BLETTERA":   "07",  # Ardèche (Bletterans → 39 Jura ?)
    "MONTBRIS":   "42",  # Loire (Montbrison)
    "AUXERRE":    "89",  # Yonne
    "CHALONS":    "51",  # Marne (Châlons-en-Champagne)
    "CHARLEV2":   "08",  # Ardennes
    "BELFORT":    "90",  # Territoire de Belfort
    "MONTELIM":   "26",  # Drôme (Montélimar)
    "RODEZ":      "12",  # Aveyron
    "TARBES":     "65",  # Hautes-Pyrénées
    "BASTIA":     "2B",  # Haute-Corse
    "TOUQUET":    "62",  # Pas-de-Calais (Le Touquet)
    "BIARRIT":    "64",  # Pyrénées-Atlantiques (Biarritz)
    "CAHORS":     "46",  # Lot
    "PERIGUEU":   "24",  # Dordogne (Périgueux)
    "EVREUX":     "27",  # Eure
    "ABBEVIL":    "80",  # Somme (Abbeville)
    "ALENCON":    "61",  # Orne
    "CHAUMONT":   "52",  # Haute-Marne
    "COLOGNAC":   "30",  # Gard
}

# Pollen clés pour les pathologies respiratoires
POLLEN_CLES = {
    "AMBROSIA":  "ambrosia",   # Ambroisie  → allergie sévère (août-sept)
    "ALNUS":     "alnus",      # Aulne/Aulnaie → allergie hiver-printemps
    "BETULA":    "betula",     # Bouleau    → allergie printemps
    "ARTEMISI":  "artemisia",  # Armoise    → allergie été
    "GRAMINE":   "graminees",  # Graminées  → allergie été (le + important)
    "GRAMINEES": "graminees",  # variante du nom
    "POACEAE":   "graminees",  # variante scientifique
    "CUPRESSU":  "cypres",     # Cyprès     → allergie hiver (Sud)
    "PLATANUS":  "platane",    # Platane    → allergie printemps (villes)
    "URTICA":    "urticacees", # Urticacées → allergie été
}

# Moisissures clés (allergènes fongiques, mêmes fichiers RNSA, colonnes séparées
# du pollen) : Alternaria et Cladosporium sont les 2 moisissures les plus
# documentées comme facteurs d'asthme/allergie sévère. Couverture partielle :
# seuls certains capteurs RNSA analysent les moisissures (ex: Lyon, Strasbourg,
# Paris) — d'autres ne mesurent que le pollen (ex: Toulouse, Clermont-Ferrand).
MOISISSURE_CLES = {
    "ALTERNAR": "alternaria",    # Alternaria   → moisissure très allergisante, asthme sévère
    "CLADOSPO": "cladosporium",  # Cladosporium → moisissure extérieure la plus répandue
}


def build_dim_pollen() -> pd.DataFrame:
    """
    Construit la table dim_pollen à partir des 253 fichiers RNSA BDD_daily.

    STRATÉGIE :
    ───────────
    1. Charger chaque fichier (ville × année)
    2. Rattacher la ville à un département via VILLE_TO_DEPT
    3. Sélectionner uniquement les colonnes de pollens clés
    4. Agréger au mois → moyenne mensuelle par département
    5. Calculer des indicateurs synthétiques de risque pollinique

    COLONNES PRODUITES :
    ────────────────────
    dept                → code département
    annee_mois          → période (ex: "2021-03")
    pollen_ambrosia_moy → concentration moyenne ambroisie (grains/m³)
    pollen_betula_moy   → concentration moyenne bouleau
    pollen_graminees_moy→ concentration moyenne graminées
    pollen_alnus_moy    → concentration moyenne aulne
    pollen_artemisia_moy→ concentration moyenne armoise
    pollen_global_max   → concentration max tous pollens confondus
    nb_jours_eleve      → nb jours avec au moins un pollen > 50 grains/m³
    moisissure_alternaria_moy   → concentration moyenne Alternaria (grains/m³)
    moisissure_cladosporium_moy → concentration moyenne Cladosporium (grains/m³)

    CLÉ PRIMAIRE : dept × annee_mois
    """

    pollen_dir = RAW_DIR / "BDD_daily_2020_2025"
    if not pollen_dir.exists():
        print(f"⚠️  Dossier manquant : {pollen_dir}")
        return pd.DataFrame()

    dfs = []
    nb_ok = 0
    nb_err = 0

    fichiers = sorted(pollen_dir.glob("*.xls"))
    print(f"Chargement de {len(fichiers)} fichiers RNSA...")

    for fpath in fichiers:
        # Extraire ville et année depuis le nom de fichier
        match = re.search(r'Particle_Extract_(.+?)_(\d{4})-\d{2}-\d{2}', fpath.name)
        if not match:
            continue

        ville = match.group(1)
        annee = int(match.group(2))

        # Rattacher au département
        dept = VILLE_TO_DEPT.get(ville)
        if dept is None:
            continue  # ville inconnue ou hors France (ex: ANDORRA)

        try:
            df = pd.read_excel(fpath, engine="xlrd")

            # Première colonne = date
            col_date = df.columns[0]
            df = df.rename(columns={col_date: "date"})
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df = df.dropna(subset=["date"])

            # Filtrage temporel
            df = df[(df["date"].dt.year >= ANNEE_DEBUT) &
                    (df["date"].dt.year <= ANNEE_FIN)]

            if df.empty:
                continue

            df["annee_mois"] = df["date"].dt.to_period("M").astype(str)
            df["dept"]       = dept

            # Identifier les colonnes de pollens clés disponibles
            cols_pollen = {}
            for col in df.columns:
                col_upper = col.upper()
                for nom_brut, nom_propre in POLLEN_CLES.items():
                    if nom_brut in col_upper:
                        df[col] = pd.to_numeric(df[col], errors="coerce")
                        cols_pollen[col] = f"pollen_{nom_propre}"
                        break

            # Identifier les colonnes de moisissures clés disponibles (mêmes
            # fichiers, colonnes distinctes des pollens — cf. MOISISSURE_CLES)
            cols_moisissure = {}
            for col in df.columns:
                col_upper = col.upper()
                for nom_brut, nom_propre in MOISISSURE_CLES.items():
                    if nom_brut in col_upper:
                        df[col] = pd.to_numeric(df[col], errors="coerce")
                        cols_moisissure[col] = f"moisissure_{nom_propre}"
                        break

            if not cols_pollen and not cols_moisissure:
                continue

            # Renommer les colonnes de pollens et moisissures
            df = df.rename(columns={**cols_pollen, **cols_moisissure})
            cols_utiles = (["dept", "annee_mois"]
                           + list(set(cols_pollen.values()))
                           + list(set(cols_moisissure.values())))
            df = df[[c for c in cols_utiles if c in df.columns]]

            # Indicateur : nb jours avec pollen élevé (> 50 grains/m³)
            pollen_cols_vals = [c for c in df.columns
                                if c.startswith("pollen_")]
            if pollen_cols_vals:
                df["jour_eleve"] = (df[pollen_cols_vals].max(axis=1) > 50).astype(int)
                df["pollen_global_max"] = df[pollen_cols_vals].max(axis=1)

            dfs.append(df)
            nb_ok += 1

        except Exception as e:
            nb_err += 1

    if not dfs:
        print("❌ Aucun fichier traité")
        return pd.DataFrame()

    print(f"✅ {nb_ok} fichiers chargés | {nb_err} erreurs")

    # Concaténer tous les fichiers
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"Total lignes brutes : {len(df_all):,}")

    # Agrégation mensuelle par département
    # Plusieurs villes peuvent appartenir au même département
    # → on fait la moyenne de toutes les stations du même dept × mois
    agg_dict = {}
    for col in df_all.columns:
        if col.startswith("pollen_") and col != "pollen_global_max":
            agg_dict[f"{col}_moy"] = (col, "mean")
    for col in df_all.columns:
        if col.startswith("moisissure_"):
            agg_dict[f"{col}_moy"] = (col, "mean")
    if "pollen_global_max" in df_all.columns:
        agg_dict["pollen_global_max"] = ("pollen_global_max", "max")
    if "jour_eleve" in df_all.columns:
        agg_dict["nb_jours_eleve"] = ("jour_eleve", "max")  # max parmi les villes du dept (pas sum qui gonfle le chiffre)

    df_agg = (
        df_all.groupby(["dept", "annee_mois"])
        .agg(**agg_dict)
        .reset_index()
    )

    # Arrondi
    for col in df_agg.select_dtypes(include="float").columns:
        df_agg[col] = df_agg[col].round(2)

    print(f"\n✅ dim_pollen : {df_agg.shape[0]:,} lignes × {df_agg.shape[1]} colonnes")
    print(f"Période      : {df_agg['annee_mois'].min()} → {df_agg['annee_mois'].max()}")
    print(f"Départements : {df_agg['dept'].nunique()} couverts")
    print(f"Colonnes     : {list(df_agg.columns)}")

    # Couverture par département
    depts_couverts = set(df_agg["dept"].unique())
    depts_manquants = set(DEPTS) - depts_couverts
    print(f"\n⚠️  {len(depts_manquants)} depts sans données pollen :")
    print(f"   {sorted(depts_manquants)}")
    print("   → Ces depts seront NaN dans df_model (normal)")

    return df_agg


# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
print("Construction de dim_pollen...")
dim_pollen = build_dim_pollen()

if not dim_pollen.empty:
    dim_pollen.to_parquet(TABLES_DIR / "dim_pollen.parquet", index=False)
    print(f"\n✅ Sauvegardé → data/processed/dim_pollen.parquet")
    display(dim_pollen.head(10))

---
## 6. 🔍 Audit de qualité — toutes les tables

In [32]:
TABLES = {
    "fact_urgences"   : TABLES_DIR / "fact_urgences.parquet",
    "dim_temps"       : TABLES_DIR / "dim_temps.parquet",
    "dim_meteo"       : TABLES_DIR / "dim_meteo.parquet",
    "dim_qualite_air" : TABLES_DIR / "dim_qualite_air.parquet",
    "dim_pollen"      : TABLES_DIR / "dim_pollen.parquet",
    "dim_geo_pop"     : TABLES_DIR / "dim_geo_pop.parquet",
    "dim_csp"         : TABLES_DIR / "dim_csp.parquet",
}

print("=" * 65)
print("AUDIT DES TABLES")
print("=" * 65)

for nom, fpath in TABLES.items():
    if not fpath.exists():
        print(f"\n❌ {nom} — fichier absent")
        continue

    df = pd.read_parquet(fpath)
    print(f"\n📋 {nom}")
    print(f"   Shape      : {df.shape[0]:>6,} lignes × {df.shape[1]} colonnes")

    # Clé primaire
    pk_candidates = [c for c in ["dept", "annee_mois", "annee"] if c in df.columns]
    if pk_candidates:
        dupes = df.duplicated(subset=pk_candidates).sum()
        print(f"   PK ({', '.join(pk_candidates)}) : {dupes} doublons {'✅' if dupes == 0 else '⚠️'}")

    # Taux de manquants
    missing = df.isnull().mean() * 100
    problemes = missing[missing > 5].round(1)
    if problemes.empty:
        print(f"   Manquants  : aucune colonne > 5% ✅")
    else:
        print(f"   Manquants  :")
        for col, pct in problemes.items():
            print(f"     {col:<35} {pct}%")

print("\n" + "=" * 65)

AUDIT DES TABLES

📋 fact_urgences
   Shape      :  6,912 lignes × 11 colonnes
   PK (dept, annee_mois) : 0 doublons ✅
   Manquants  :
     taux_sos_allergie                   53.5%
     taux_sos_asthme                     53.5%
     taux_sos_bronchiolite               53.6%

📋 dim_temps
   Shape      :     72 lignes × 13 colonnes
   PK (annee_mois, annee) : 0 doublons ✅
   Manquants  : aucune colonne > 5% ✅

📋 dim_meteo
   Shape      :  2,883 lignes × 8 colonnes
   PK (dept, annee_mois) : 0 doublons ✅
   Manquants  : aucune colonne > 5% ✅

📋 dim_qualite_air
   Shape      :  2,518 lignes × 9 colonnes
   PK (dept, annee_mois) : 0 doublons ✅
   Manquants  :
     o3_moy                              11.0%
     so2_moy                             44.3%

📋 dim_pollen
   Shape      :  1,686 lignes × 11 colonnes
   PK (dept, annee_mois) : 0 doublons ✅
   Manquants  :
     pollen_platane_moy                  41.4%
     pollen_graminees_moy                32.1%
     pollen_ambrosia_moy           

---
## 7. 🔗 Vue analytique — `df_model` (jointure pour la modélisation)

Pour XGBoost ou LSTM, on a besoin d'un DataFrame plat.  
Cette cellule joint toutes les tables en un seul `df_model` sans toucher aux tables sources.

In [33]:
def build_model_view() -> pd.DataFrame:
    """
    Crée la vue analytique plate pour la modélisation.
    Jointure : fact_urgences ← dim_temps ← dim_meteo ← dim_geo_pop ← dim_csp
    """
    if not (TABLES_DIR / "fact_urgences.parquet").exists():
        print("⚠️  fact_urgences.parquet manquant — exécutez d'abord les cellules 1 à 5")
        return pd.DataFrame()

    df = pd.read_parquet(TABLES_DIR / "fact_urgences.parquet")

    # Jointure dim_temps (annee_mois → enrichissement temporel)
    if (TABLES_DIR / "dim_temps.parquet").exists():
        dt = pd.read_parquet(TABLES_DIR / "dim_temps.parquet")
        df = df.merge(dt, on="annee_mois", how="left")
        print("✅ dim_temps joint")

    # Jointure dim_meteo (dept × annee_mois)
    if (TABLES_DIR / "dim_meteo.parquet").exists():
        dm = pd.read_parquet(TABLES_DIR / "dim_meteo.parquet")
        df = df.merge(dm, on=["dept", "annee_mois"], how="left")
        print("✅ dim_meteo joint")

    # Jointure dim_geo_pop (dept)
    if (TABLES_DIR / "dim_geo_pop.parquet").exists():
        dg = pd.read_parquet(TABLES_DIR / "dim_geo_pop.parquet")
        df = df.merge(dg, on="dept", how="left")
        print("✅ dim_geo_pop joint")

    # Jointure dim_csp (dept)
    if (TABLES_DIR / "dim_csp.parquet").exists():
        dc = pd.read_parquet(TABLES_DIR / "dim_csp.parquet")
        df = df.merge(dc, on="dept", how="left")
        print("✅ dim_csp joint")

    # Jointure dim_qualite_air (dept × annee_mois)
    if (TABLES_DIR / "dim_qualite_air.parquet").exists():
        dqa = pd.read_parquet(TABLES_DIR / "dim_qualite_air.parquet")
        df = df.merge(dqa, on=["dept", "annee_mois"], how="left")
        print("✅ dim_qualite_air joint")

    # Jointure dim_pollen (dept × annee_mois)
    if (TABLES_DIR / "dim_pollen.parquet").exists():
        dp = pd.read_parquet(TABLES_DIR / "dim_pollen.parquet")
        df = df.merge(dp, on=["dept", "annee_mois"], how="left")
        print("✅ dim_pollen joint")

    # Vérification de couverture : les colonnes de taux existent-elles ?
    cols_taux_urgences = [c for c in df.columns if c.startswith("taux_urgences_")]
    if not cols_taux_urgences:
        print("⚠️  Aucune colonne taux_urgences_* trouvée dans fact_urgences")

    # Résumé du taux de remplissage par groupe de variables
    groupes = {
        "Y (urgences)": [c for c in df.columns if c.startswith("taux_urgences_")],
        "Météo":        [c for c in df.columns if c in ["temp_moy","temp_min","temp_max","humidite_moy","vent_moy"]],
        "Air quality":  [c for c in df.columns if c.startswith("atmo_") or (c.endswith("_moy") and c[:2] in ["pm","no","o3"])],
        "Pollen":       [c for c in df.columns if c.startswith("pollen_") or c == "nb_semaines_elevee"],
        "Démographie":  [c for c in df.columns if c in ["pop_totale","part_seniors","part_jeunes","tx_urbain"]],
    }
    print("\nTaux de remplissage par groupe :")
    for groupe, cols in groupes.items():
        if cols:
            pct = df[cols].notna().mean().mean() * 100
            bar = '█' * int(pct // 10) + '░' * (10 - int(pct // 10))
            print(f"  {groupe:<18} {bar} {pct:.0f}%")

    print(f"\n📦 df_model : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
    return df


df_model = build_model_view()

if not df_model.empty:
    df_model.to_parquet(TABLES_DIR / "df_model.parquet", index=False)
    df_model.to_csv(TABLES_DIR / "df_model.csv", index=False)
    print(f"\n✅ Vue analytique sauvegardée : {TABLES_DIR / 'df_model.parquet'}")
    print("\nColonnes disponibles pour la modélisation :")
    print(list(df_model.columns))
    display(df_model.head(5))

✅ dim_temps joint
✅ dim_meteo joint
✅ dim_geo_pop joint
✅ dim_csp joint
✅ dim_qualite_air joint
✅ dim_pollen joint

Taux de remplissage par groupe :
  Y (urgences)       █████████░ 100%
  Météo              ████░░░░░░ 42%
  Air quality        ███░░░░░░░ 35%
  Pollen             █░░░░░░░░░ 15%
  Démographie        ██████████ 100%

📦 df_model : 6,912 lignes × 58 colonnes

✅ Vue analytique sauvegardée : data/tables/df_model.parquet

Colonnes disponibles pour la modélisation :
['dept', 'annee_mois', 'taux_urgences_allergie', 'taux_hosp_allergie', 'taux_sos_allergie', 'taux_urgences_asthme', 'taux_hosp_asthme', 'taux_sos_asthme', 'taux_urgences_bronchiolite', 'taux_hosp_bronchiolite', 'taux_sos_bronchiolite', 'annee', 'mois', 'trimestre', 'semestre', 'sin_mois', 'cos_mois', 'est_hiver', 'est_printemps', 'est_ete', 'est_automne', 'saison_pollen', 'flag_covid', 'temp_moy', 'temp_min', 'temp_max', 'humidite_moy', 'vent_moy', 'precip_total', 'pop_totale', 'part_seniors', 'part_jeunes', 'tx_urba

,dept,annee_mois,taux_urgences_allergie,taux_hosp_allergie,taux_sos_allergie,taux_urgences_asthme,taux_hosp_asthme,taux_sos_asthme,taux_urgences_bronchiolite,taux_hosp_bronchiolite,...,nb_jours_pm10_eleve,pollen_platane_moy,pollen_graminees_moy,pollen_ambrosia_moy,pollen_artemisia_moy,pollen_alnus_moy,pollen_betula_moy,pollen_urticacees_moy,pollen_global_max,nb_jours_eleve
0,01,2020-01,815.39,245.12,NaN,402.03,878.26,NaN,23287.66,44861.11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
1,01,2020-02,722.06,228.66,NaN,462.67,437.93,NaN,10169.56,25875.35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
2,01,2020-03,377.48,193.97,NaN,805.16,841.07,NaN,9939.19,14230.77,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,01,2020-04,500.03,98.04,NaN,574.89,863.81,NaN,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
4,01,2020-05,568.77,311.94,NaN,522.41,552.69,NaN,0.00,0.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


---
## 📁 Récapitulatif des fichiers produits

```
data/
├── raw/                                      ← Fichiers bruts téléchargés manuellement
│   ├── allergie_urgences.csv                 ← Santé Publique France
│   ├── asthme_urgences.csv
│   ├── bronchiolite_urgences.csv
│   ├── MeteoFrance/
│   │   ├── MENSQ_01_previous-1950-2024.csv.gz ← Météo France, données mensuelles (1 fichier/dept)
│   │   ├── MENSQ_01_latest-2025-2026.csv.gz
│   │   └── ... (95 depts × 2 périodes = 190 fichiers)
│   ├── AASQA/
│   │   ├── FR_E2_2021-01-01.csv               ← Qualité de l'air AASQA (mensuel)
│   │   ├── ... (60 fichiers, 2021-2025)
│   │   └── stations_metadata.xls              ← Métadonnées stations (lat/lon → dept)
│   ├── BDD_daily_2020_2025/
│   │   ├── Particle_Extract_AGEN_2020-...xls ← Pollen RNSA (253 fichiers)
│   │   └── ...
│   └── geo_pop/
│       ├── insee_demographie.csv             ← CNAM Cartographie des pathologies (2015-2024)
│       ├── insee_urbanisation.xlsx
│       ├── ameli_acces_soins_2020.xls
│       ├── ameli_acces_soins_2021.xls
│       ├── ameli_acces_soins_2022.xls
│       ├── ameli_acces_soins_2023.xls
│       └── ameli_acces_soins_2024.xls
│   └── insee_csp/
│       └── insee_csp.xlsx
│
└── tables/                                   ← Tables produites par ce notebook
├── stations_aasqa_dept.parquet           ← Correspondance station AASQA → dept
├── dim_temps.parquet                     ← 72 mois, variables temporelles + saisonnalité
├── fact_urgences.parquet                 ← 6 912 lignes, 3 pathologies × 3 indicateurs
├── dim_meteo.parquet                     ← 6 912 lignes, 96 depts × 72 mois
├── dim_qualite_air.parquet               ← ~5 516 lignes, 92 depts × 60 mois (2021-2025)
├── dim_pollen.parquet                    ← 1 686 lignes, 49 depts × 3 années
├── dim_geo_pop.parquet                   ← 96 depts, démographie + urbanisation + soins
├── dim_csp.parquet                       ← 96 depts, 7 catégories socioprofessionnelles
├── df_model.parquet                      ← 6 912 lignes × 61 colonnes (vue analytique)
└── df_model.csv                          ← Idem, format CSV
```
### Ordre d'exécution
| Étape | Table | Données requises | Lignes produites |
|---|---|---|---|
| 0 | Configuration | — | — |
| 1 | `dim_temps` | Aucune (calculé) | 72 |
| 2 | `fact_urgences` | allergie / asthme / bronchiolite CSV | 6 912 |
| 3 | `dim_meteo` | MeteoFrance/MENSQ_*.csv.gz | 6 912 |
| 4 | `dim_geo_pop` | insee_demographie + urbanisation + ameli | 96 |
| 5 | `dim_csp` | insee_csp.xlsx (feuille "DEP") | 96 |
| 6A | `dim_qualite_air` | AASQA/FR_E2_*.csv + stations_metadata.xls | ~5 516 |
| 6B | `dim_pollen` | BDD_daily_2020_2025/*.xls | 1 686 |
| 6 | Audit qualité | Toutes les tables ci-dessus | — |
| 7 | `df_model` | Toutes les tables ci-dessus | 6 912 × 61 col |

### Limites connues
| Variable | Statut | Raison |
|---|---|---|
| Qualité de l'air 2020 | ❌ Absent | Flux temps réel LCSQA non archivé avant 2021 |
| Qualité de l'air : SO₂ | ⚠️ ~28% | Stations de mesure SO₂ peu nombreuses (limite réseau, pas un bug) |
| Qualité de l'air : 4 depts (09, 11, 46, 48) | ❌ Absent | Aucune station AASQA géocodée dans ces depts, ou aucune mesure exploitable (cf. §6A) |
| Pollen 2022, 2024, 2025 | ❌ Absent | Non publiés par le RNSA dans BDD_daily |
| Pollen : ~40% des capteurs (37/93 villes) | ❌ Ignorés silencieusement | `VILLE_TO_DEPT` incomplet/mal orthographié — pas de coordonnées dans les fichiers RNSA pour corriger par reverse geocoding (cf. §6B). Non corrigé (décision : coût jugé non prioritaire). |
| Pollen : `nb_jours_eleve` | ⚠️ Toujours 0 ou 1 | `.max()` sur un flag binaire au lieu d'un décompte de jours — nom de colonne trompeur (cf. §6B). Non corrigé. |
| RNSA (source pollen) | ⚠️ Organisme liquidé le 26/03/2025 | Aucune nouvelle donnée à venir ; alternative Atmo France inutilisable pour 2020-2025 (cf. §6B) |

### Journal des modifications apportées au pipeline (session de revue)
| # | Table / section | Type | Détail |
|---|---|---|---|
| 1 | `fact_urgences` (docstring `parse_urgences`) | 📝 Documentation corrigée | La définition documentée était fausse ("pour 100k hab" = taux de population). Corrigée : `taux_urgences/hosp/sos` sont des **parts relatives** (dénominateur = volume total toutes causes du même sous-groupe dept/classe d'âge/semaine), pas des taux d'incidence en population. Ajout d'un avertissement : ne jamais agréger entre départements ni entre classes d'âge. |
| 2 | `dim_meteo` (§3) | 🔧 Bug corrigé (source remplacée) | SYNOP ne couvrait que 41/96 depts (42 stations). Remplacé par Météo France MENSQ (mensuel, 1 fichier par département, 190 fichiers téléchargés) → **96/96 depts**, `temp_moy` etc. passent de 58% à 0% de NaN. Corse (code "20") séparée en 2A/2B par reverse geocoding (`split_corse_2a_2b`). |
| 3 | `dim_geo_pop.pop_totale` (§4) | 🔧 Bug corrigé | `Npop` sommé sur tous les `cla_age_5` **y compris** "tsage" (déjà l'agrégat "tous âges") → population comptée ~2x (ex : Paris à 4,14M au lieu de 2,07M ; total France à 130M au lieu de ~65M). Corrigé en utilisant directement la ligne `cla_age_5 == "tsage"`. `part_seniors`/`part_jeunes` corrigés du même coup (ils étaient sous-estimés de moitié). |
| 4 | `dim_geo_pop.prev_resp_chronique` (§4) | ➕ Variable ajoutée | Nouvelle colonne : prévalence des maladies respiratoires chroniques (hors mucoviscidose) par département, dernière année dispo (2024). Proxy de vulnérabilité respiratoire de fond — ne remplace pas `taux_urgences_*`, aucune catégorie équivalente n'existe pour allergie/bronchiolite dans la nomenclature CNAM. |
| 5 | `insee_demographie.csv` (§4) | 🔄 Donnée rafraîchie | Re-téléchargé : passait de 2015-2023 à **2015-2024** (mise à jour CNAM du 06/07/2026). Impact vérifié négligeable sur `prev_resp_chronique` (corrélation inter-année 0,998, +0,34 pt en moyenne). |
| 6 | `dim_qualite_air` (§6A) | 🔧 Bug corrigé | Département extrait à tort du préfixe numérique du `code site` (non géographique — ex : `FR01011` = Metz/57, pas Ain/01). Corrigé par reverse geocoding des vraies coordonnées (nouveau fichier `stations_metadata.xls` + nouvelle fonction `build_station_dict_aasqa()`) → couverture **42/96 → 92/96 depts**, `no2_moy`/`pm10_moy` de ~36% à ~97-98%. |
| 7 | `dim_pollen` (§6B) | 📝 Documenté, non corrigé | ~40% des capteurs (37/93 villes) silencieusement ignorés par `VILLE_TO_DEPT` (typos/entrées manquantes) ; `nb_jours_eleve` toujours égal à 0 ou 1 (bug d'agrégation). Pas de coordonnées dans les fichiers RNSA → pas de fix automatique possible comme pour météo/AASQA. Alternative (indice pollen Atmo France) vérifiée et écartée (pas d'historique avant 2026, indice catégoriel). Décision : documenté comme limite connue, non corrigé pour l'instant. |

| 8 | `dim_pollen` (§6B) | ➕ Variables ajoutées | `moisissure_alternaria_moy` et `moisissure_cladosporium_moy` — mêmes fichiers RNSA, colonnes fongiques jusque-là non extraites. Couverture station : 66/93 villes (71%) ; couverture dept×mois dans `dim_pollen` : ~26% (même limite de fond que le reste du pollen). |

À chaque modification, les tables `.parquet` concernées ainsi que `df_model.parquet` /
`df_model.csv` ont été régénérées pour rester cohérentes.
